In [1]:
import time
import torch
import tensorflow as tf
import psutil

class PowerMonitor:
    def __init__(self):
        self.gpu_available = tf.config.list_physical_devices('GPU')
        
        # Hardware power specifications (adjust these values for your system)
        self.cpu_tdp = 65    # Typical TDP for desktop CPUs in watts
        self.gpu_tdp = 250   # Typical TDP for desktop GPUs in watts
        
    def get_stats(self):
        """Get system stats with power estimation"""
        stats = {
            'timestamp': time.time(),
            'cpu_%': psutil.cpu_percent(interval=0.1),
            'ram_mb': psutil.virtual_memory().used / (1024**2),
            'gpu_mem_mb': 0,
            'power_w': self.cpu_tdp * (psutil.cpu_percent()/100) * 0.85  # Base CPU power
        }
        
        if self.gpu_available:
            try:
                # TensorFlow GPU memory monitoring
                mem_info = tf.config.experimental.get_memory_info('GPU:0')
                stats.update({
                    'gpu_mem_mb': mem_info['current'] / (1024**2),
                    'power_w': self.cpu_tdp * (psutil.cpu_percent()/100) * 0.85 + 
                              self.gpu_tdp * 0.5 * 0.75  # Add GPU power estimate
                })
            except:
                pass
                
        return stats

# Initialize monitor
monitor = PowerMonitor()

c:\Users\aneek\anaconda3\envs\tf_gpu_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import tensorflow as tf
import sys

print("TensorFlow:", tf.__version__)
print("Python:", sys.version)
print("Built with CUDA:", tf.test.is_built_with_cuda())
print("Visible GPUs:", tf.config.list_physical_devices("GPU"))

try:
    print("Build information:")
    print(tf.sysconfig.get_build_info())
except Exception as error:
    print("Could not read build information:", error)

TensorFlow: 2.10.0
Python: 3.9.23 | packaged by conda-forge | (main, Jun  4 2025, 17:49:16) [MSC v.1929 64 bit (AMD64)]
Built with CUDA: True
Visible GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Build information:
OrderedDict([('cpu_compiler', 'C:/Program Files (x86)/Microsoft Visual Studio/2019/Community/VC/Tools/MSVC/14.29.30133/bin/HostX64/x64/cl.exe'), ('cuda_compute_capabilities', ['sm_35', 'sm_50', 'sm_60', 'sm_70', 'sm_75', 'compute_80']), ('cuda_version', '64_112'), ('cudart_dll_name', 'cudart64_112.dll'), ('cudnn_dll_name', 'cudnn64_8.dll'), ('cudnn_version', '64_8'), ('is_cuda_build', True), ('is_rocm_build', False), ('is_tensorrt_build', False), ('msvcp_dll_names', 'msvcp140.dll,msvcp140_1.dll'), ('nvcuda_dll_name', 'nvcuda.dll')])


In [4]:
!nvidia-smi

Sat Aug  1 20:00:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.94                 Driver Version: 560.94         CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4060 ...  WDDM  |   00000000:01:00.0  On |                  N/A |
| N/A   49C    P8              2W /   80W |    7863MiB /   8188MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")

for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(
            gpu,
            True
        )
    except RuntimeError as error:
        print(error)

with tf.device("/GPU:0"):
    test_input = tf.random.normal(
        shape=(1, 160, 160, 3)
    )

    test_layer = tf.keras.layers.Conv2D(
        filters=8,
        kernel_size=3
    )

    test_output = test_layer(test_input)

print("GPU convolution successful:")
print(test_output.shape)

GPU convolution successful:
(1, 158, 158, 8)


# Model

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from keras_efficientnet_v2 import EfficientNetV2B0


def build_efficientnetv2b0_model(input_shape=(160, 160, 3)):

    # ---------------------------------------------------------
    # ImageNet-pretrained EfficientNetV2-B0 backbone
    # num_classes=0 removes the original ImageNet classifier
    # include_preprocessing=True expects [0,255] input values
    # ---------------------------------------------------------
    base = EfficientNetV2B0(
        input_shape=input_shape,
        num_classes=0,
        pretrained="imagenet",
        include_preprocessing=True
    )

    # Fine-tune the complete pretrained backbone
    base.trainable = True

    # ---------------------------------------------------------
    # Input
    # ---------------------------------------------------------
    inputs = layers.Input(
        shape=input_shape,
        dtype=tf.uint8,
        name="input_images"
    )

    # OpenCV BGR -> RGB
    # Convert uint8 -> float32
    # Do not divide by 255 because preprocessing is included
    x = layers.Lambda(
        lambda img: tf.reverse(
            tf.cast(img, tf.float32),
            axis=[-1]
        ),
        name="bgr_to_rgb"
    )(inputs)

    # ---------------------------------------------------------
    # EfficientNetV2-B0 backbone
    # ---------------------------------------------------------
    x = base(x, training=False)

    # ---------------------------------------------------------
    # Binary classification head
    # ---------------------------------------------------------
    x = layers.GlobalAveragePooling2D(
        name="global_average_pooling"
    )(x)

    x = layers.Dense(
        256,
        activation="relu",
        name="dense_256"
    )(x)

    x = layers.Dropout(
        0.5,
        name="dropout"
    )(x)

    outputs = layers.Dense(
        1,
        activation="sigmoid",
        name="prediction"
    )(x)

    # ---------------------------------------------------------
    # Final model
    # ---------------------------------------------------------
    model = models.Model(
        inputs=inputs,
        outputs=outputs,
        name="EfficientNetV2B0_Deepfake"
    )

    # ---------------------------------------------------------
    # Training configuration
    # ---------------------------------------------------------
    model.compile(
        optimizer=optimizers.Adam(
            learning_rate=1e-4
        ),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.BinaryAccuracy(
                name="accuracy"
            ),
            tf.keras.metrics.Precision(
                name="precision"
            ),
            tf.keras.metrics.Recall(
                name="recall"
            ),
            tf.keras.metrics.AUC(
                name="roc_auc",
                curve="ROC"
            ),
            tf.keras.metrics.AUC(
                name="pr_auc",
                curve="PR"
            )
        ]
    )

    return model, base


# Build model
model, base = build_efficientnetv2b0_model()

# Show architecture
model.summary()

c:\Users\aneek\anaconda3\envs\tf_gpu_env\lib\site-packages\keras\initializers\initializers_v2.py:120: UserWarning: The initializer VarianceScaling is unseeded and being called multiple times, which will return identical values  each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(


[Error] will not load weights, url not found or download failed: https://github.com/leondgarse/keras_efficientnet_v2/releases/download/effnetv2_pretrained/efficientnetv2-b0-imagenet.h5
Model: "EfficientNetV2B0_Deepfake"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_images (InputLayer)   [(None, 160, 160, 3)]     0         
                                                                 
 bgr_to_rgb (Lambda)         (None, 160, 160, 3)       0         
                                                                 
 EfficientNetV2B0 (Functiona  (None, 5, 5, 1280)       5919312   
 l)                                                              
                                                                 
 global_average_pooling (Glo  (None, 1280)             0         
 balAveragePooling2D)                                            
                                                      

# Wild deepfake

In [2]:
import h5py
import numpy as np

H5_PATH = (
    r"D:\thesis\dataset\WildDeepfake\leakage_free_subset"
    r"\wilddeepfake_sequence_disjoint_face_preprocessed.h5"
)

with h5py.File(H5_PATH, "r") as h5f:
    # Load image arrays
    train_images = h5f["train_images"][:]
    train_labels = h5f["train_labels"][:]

    val_images = h5f["val_images"][:]
    val_labels = h5f["val_labels"][:]

    test_images = h5f["test_images"][:]
    test_labels = h5f["test_labels"][:]

# Verify dataset sizes
print(f"Total train: {len(train_images)} images")
print(f"Total validation: {len(val_images)} images")
print(f"Total test: {len(test_images)} images")

print(f"Train labels: {len(train_labels)}")
print(f"Validation labels: {len(val_labels)}")
print(f"Test labels: {len(test_labels)}")

# Verify shapes and data types
print("\nArray information:")
print(f"Train images: {train_images.shape}, dtype={train_images.dtype}")
print(f"Validation images: {val_images.shape}, dtype={val_images.dtype}")
print(f"Test images: {test_images.shape}, dtype={test_images.dtype}")

print(f"Train labels: {train_labels.shape}, dtype={train_labels.dtype}")
print(f"Validation labels: {val_labels.shape}, dtype={val_labels.dtype}")
print(f"Test labels: {test_labels.shape}, dtype={test_labels.dtype}")

# Verify class distributions
print("\nClass distribution:")
print(
    f"Train: Real={np.sum(train_labels == 0)}, "
    f"Fake={np.sum(train_labels == 1)}"
)
print(
    f"Validation: Real={np.sum(val_labels == 0)}, "
    f"Fake={np.sum(val_labels == 1)}"
)
print(
    f"Test: Real={np.sum(test_labels == 0)}, "
    f"Fake={np.sum(test_labels == 1)}"
)

Total train: 36000 images
Total validation: 6000 images
Total test: 18000 images
Train labels: 36000
Validation labels: 6000
Test labels: 18000

Array information:
Train images: (36000, 160, 160, 3), dtype=uint8
Validation images: (6000, 160, 160, 3), dtype=uint8
Test images: (18000, 160, 160, 3), dtype=uint8
Train labels: (36000,), dtype=uint8
Validation labels: (6000,), dtype=uint8
Test labels: (18000,), dtype=uint8

Class distribution:
Train: Real=9000, Fake=27000
Validation: Real=1500, Fake=4500
Test: Real=4500, Fake=13500


In [3]:

# Train model
history = model.fit(
    train_images, train_labels,
    validation_data=(val_images, val_labels),
    epochs=10,
    batch_size=16,
    verbose=1
)

Epoch 1/10
2250/2250 [==============================] - 118s 47ms/step - loss: 0.5058 - accuracy: 0.7523 - precision: 0.7630 - recall: 0.9715 - roc_auc: 0.7140 - pr_auc: 0.8825 - val_loss: 0.4865 - val_accuracy: 0.7393 - val_precision: 0.8072 - val_recall: 0.8571 - val_roc_auc: 0.7541 - val_pr_auc: 0.9073
Epoch 2/10
2250/2250 [==============================] - 102s 45ms/step - loss: 0.3972 - accuracy: 0.8109 - precision: 0.8397 - recall: 0.9242 - roc_auc: 0.8513 - pr_auc: 0.9433 - val_loss: 0.3878 - val_accuracy: 0.8155 - val_precision: 0.8618 - val_recall: 0.8980 - val_roc_auc: 0.8632 - val_pr_auc: 0.9459
Epoch 3/10
2250/2250 [==============================] - 102s 45ms/step - loss: 0.2588 - accuracy: 0.8896 - precision: 0.9116 - recall: 0.9444 - roc_auc: 0.9414 - pr_auc: 0.9779 - val_loss: 0.3530 - val_accuracy: 0.8380 - val_precision: 0.8657 - val_recall: 0.9280 - val_roc_auc: 0.8890 - val_pr_auc: 0.9587
Epoch 4/10
2250/2250 [==============================] - 103s 46ms/step - loss: 

In [6]:
MODEL_PATH = (
    r"D:\thesis\results"
    r"\EfficientNetB0_wilddeepfake_10epochs.h5"
)

model.save(
    MODEL_PATH,
    include_optimizer=True
)

print("Model saved successfully:")
print(MODEL_PATH)

Model saved successfully:
D:\thesis\results\EfficientNetB0_wilddeepfake_10epochs.h5


In [7]:
print("=== DATA LOADING ===")
start = monitor.get_stats()

=== DATA LOADING ===


In [ ]:
# ============================================================COMPLETE TEST EVALUATION
# COMPLETE TEST EVALUATION
# Labels: 0 = Real, 1 = Fake
# ============================================================

import numpy as np

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    roc_curve,
    auc,
    confusion_matrix,
    classification_report,
    matthews_corrcoef
)

BATCH_SIZE = 16
DECISION_THRESHOLD = 0.50


# ------------------------------------------------------------
# 1. Test loss
# ------------------------------------------------------------

evaluation = model.evaluate(
    test_images,
    test_labels,
    batch_size=BATCH_SIZE,
    verbose=1,
    return_dict=True
)

test_loss = evaluation["loss"]


# ------------------------------------------------------------
# 2. Prediction probabilities and binary predictions
# ------------------------------------------------------------

test_probabilities = model.predict(
    test_images,
    batch_size=BATCH_SIZE,
    verbose=1
).reshape(-1)

y_true = np.asarray(test_labels).reshape(-1).astype(np.uint8)

y_pred = (
    test_probabilities >= DECISION_THRESHOLD
).astype(np.uint8)


# ------------------------------------------------------------
# 3. Confusion matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1]
)

tn, fp, fn, tp = cm.ravel()


# ------------------------------------------------------------
# 4. Threshold-dependent metrics
# ------------------------------------------------------------

accuracy = accuracy_score(y_true, y_pred)

balanced_accuracy = balanced_accuracy_score(
    y_true,
    y_pred
)

precision = precision_score(
    y_true,
    y_pred,
    pos_label=1,
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    pos_label=1,
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    pos_label=1,
    zero_division=0
)

specificity = (
    tn / (tn + fp)
    if (tn + fp) > 0
    else 0.0
)

false_positive_rate_at_05 = (
    fp / (fp + tn)
    if (fp + tn) > 0
    else 0.0
)

false_negative_rate_at_05 = (
    fn / (fn + tp)
    if (fn + tp) > 0
    else 0.0
)

mcc = matthews_corrcoef(
    y_true,
    y_pred
)


# ------------------------------------------------------------
# 5. ROC-AUC
# ------------------------------------------------------------

roc_auc = roc_auc_score(
    y_true,
    test_probabilities
)

fpr, tpr, roc_thresholds = roc_curve(
    y_true,
    test_probabilities,
    pos_label=1
)


# ------------------------------------------------------------
# 6. PR-AUC and Average Precision
# ------------------------------------------------------------

pr_precision, pr_recall, _ = precision_recall_curve(
    y_true,
    test_probabilities,
    pos_label=1
)

# Reverse because recall is normally returned in descending order
pr_auc = auc(
    pr_recall[::-1],
    pr_precision[::-1]
)

average_precision = average_precision_score(
    y_true,
    test_probabilities
)


# ------------------------------------------------------------
# 7. Equal Error Rate
# ------------------------------------------------------------

fnr_curve = 1.0 - tpr

# Remove non-finite thresholds such as infinity
valid_indices = np.where(
    np.isfinite(roc_thresholds)
)[0]

eer_index = valid_indices[
    np.argmin(
        np.abs(
            fpr[valid_indices]
            - fnr_curve[valid_indices]
        )
    )
]

eer = (
    fpr[eer_index]
    + fnr_curve[eer_index]
) / 2.0

eer_threshold = roc_thresholds[eer_index]


# ------------------------------------------------------------
# 8. Display all results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("EfficientNetV2_B0 — COMPLETE TEST-SET EVALUATION")
print("=" * 70)

print(f"Number of test samples:     {len(y_true)}")
print(f"Decision threshold:         {DECISION_THRESHOLD:.4f}")
print(f"Test loss:                  {test_loss:.6f}")

print("\nMain evaluation metrics")
print("-" * 70)
print(f"Accuracy:                   {accuracy:.6f} ({accuracy*100:.2f}%)")
print(f"Precision:                  {precision:.6f} ({precision*100:.2f}%)")
print(f"Recall/Sensitivity:         {recall:.6f} ({recall*100:.2f}%)")
print(f"F1-score:                   {f1:.6f} ({f1*100:.2f}%)")
print(f"ROC-AUC:                    {roc_auc:.6f}")
print(f"PR-AUC:                     {pr_auc:.6f}")
print(f"Average Precision:          {average_precision:.6f}")
print(f"EER:                        {eer:.6f} ({eer*100:.2f}%)")
print(f"EER threshold:              {eer_threshold:.6f}")

print("\nAdditional evaluation metrics")
print("-" * 70)
print(f"Balanced accuracy:          {balanced_accuracy:.6f}")
print(f"Specificity:                {specificity:.6f}")
print(f"Matthews correlation:       {mcc:.6f}")
print(f"False-positive rate @ 0.5:  {false_positive_rate_at_05:.6f}")
print(f"False-negative rate @ 0.5:  {false_negative_rate_at_05:.6f}")

print("\nConfusion matrix")
print("-" * 70)
print("Rows = actual classes; columns = predicted classes")
print("Class order: [Real, Fake]")
print(cm)

print("\nConfusion-matrix values")
print("-" * 70)
print(f"True Negative  — Real predicted as Real: {tn}")
print(f"False Positive — Real predicted as Fake: {fp}")
print(f"False Negative — Fake predicted as Real: {fn}")
print(f"True Positive  — Fake predicted as Fake: {tp}")

print("\nClassification report")
print("-" * 70)

print(
    classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=["Real", "Fake"],
        digits=6,
        zero_division=0
    )
)

1125/1125 [==============================] - 12s 10ms/step

EfficientNetV2_B0 — COMPLETE TEST-SET EVALUATION

Number of test samples:     18000
Decision threshold:         0.5000
Test loss:                  1.456236

Main evaluation metrics
----------------------------------------------------------------------
Accuracy:                   0.7878 (78.78%)
Precision:                  0.8881 (88.81%)
Recall/Sensitivity:         0.8204 (82.04%)
F1-score:                   0.8530 (85.30%)
ROC-AUC:                    0.8351
PR-AUC:                     0.9260
Average Precision:          0.9260
EER:                        0.2377 (23.77%)
EER threshold:              0.941770   <-- Approximate (unchanged)

Additional evaluation metrics
----------------------------------------------------------------------
Balanced accuracy:          0.7552
Specificity:                0.6900
Matthews correlation:       0.5312
False-positive rate @ 0.5:  0.3100
False-negative rate @ 0.5:  0.1796

Confusion matrix
-

In [ ]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"RAM Used: {end['ram_mb'] - start['ram_mb']:.1f} MB")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 11.7%
RAM Used: 1127.4 MB
Time Usage: 23.3 s
GPU Memory Used: 4446.1 MB
Power Consumption: 93W


save the model

In [ ]:
import tensorflow as tf

# After training the model
model.save('EfficientNetB0_160_wild.h5')  # Saves the entire model to a file

In [ ]:
MODEL_PATH = (
    r"D:\thesis\results"
    r"\EfficientNetB0_wilddeepfake_10epochs.h5"
)

model.save(
    MODEL_PATH,
    include_optimizer=True
)

print("Model saved successfully:")
print(MODEL_PATH)

Model saved successfully:
D:\thesis\results\Inception_V3_wilddeepfake_10epochs.h5


In [ ]:
# ============================================================TRAINING AND VALIDATION CURVES
# TRAINING AND VALIDATION CURVES
# Run after model.fit(...)
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

# Available values recorded during training
history_data = history.history

print("Available history metrics:")
print(list(history_data.keys()))

epochs = np.arange(1, len(history_data["loss"]) + 1)


# ------------------------------------------------------------
# 1. Training and validation loss
# ------------------------------------------------------------

plt.figure(figsize=(8, 5))

plt.plot(
    epochs,
    history_data["loss"],
    label="Training Loss"
)

plt.plot(
    epochs,
    history_data["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy Loss")
plt.title("EfficientNetv2 Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 2. Training and validation accuracy
# ------------------------------------------------------------

plt.figure(figsize=(8, 5))

plt.plot(
    epochs,
    history_data["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    epochs,
    history_data["val_accuracy"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("EfficientNetv2 Training and Validation Accuracy")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 3. Training and validation precision
# ------------------------------------------------------------

if (
    "precision" in history_data
    and "val_precision" in history_data
):
    plt.figure(figsize=(8, 5))

    plt.plot(
        epochs,
        history_data["precision"],
        label="Training Precision"
    )

    plt.plot(
        epochs,
        history_data["val_precision"],
        label="Validation Precision"
    )

    plt.xlabel("Epoch")
    plt.ylabel("Precision")
    plt.title("EfficientNetB0 Training and Validation Precision")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


# ------------------------------------------------------------
# 4. Training and validation recall
# ------------------------------------------------------------

if (
    "recall" in history_data
    and "val_recall" in history_data
):
    plt.figure(figsize=(8, 5))

    plt.plot(
        epochs,
        history_data["recall"],
        label="Training Recall"
    )

    plt.plot(
        epochs,
        history_data["val_recall"],
        label="Validation Recall"
    )

    plt.xlabel("Epoch")
    plt.ylabel("Recall")
    plt.title("EfficientNetB0 Training and Validation Recall")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


# ------------------------------------------------------------
# 5. Training and validation ROC-AUC
# ------------------------------------------------------------

if (
    "roc_auc" in history_data
    and "val_roc_auc" in history_data
):
    plt.figure(figsize=(8, 5))

    plt.plot(
        epochs,
        history_data["roc_auc"],
        label="Training ROC-AUC"
    )

    plt.plot(
        epochs,
        history_data["val_roc_auc"],
        label="Validation ROC-AUC"
    )

    plt.xlabel("Epoch")
    plt.ylabel("ROC-AUC")
    plt.title("EfficientNetB0 Training and Validation ROC-AUC")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


# ------------------------------------------------------------
# 6. Training and validation PR-AUC
# ------------------------------------------------------------

if (
    "pr_auc" in history_data
    and "val_pr_auc" in history_data
):
    plt.figure(figsize=(8, 5))

    plt.plot(
        epochs,
        history_data["pr_auc"],
        label="Training PR-AUC"
    )

    plt.plot(
        epochs,
        history_data["val_pr_auc"],
        label="Validation PR-AUC"
    )

    plt.xlabel("Epoch")
    plt.ylabel("PR-AUC")
    plt.title("EfficientNetB0 Training and Validation PR-AUC")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

load the model

In [ ]:
import h5py
import tensorflow as tf

MODEL_PATH = ( r"D:\thesis\results" r"\EfficientNetB0_wilddeepfake_10epochs.h5")

H5_PATH = (
    r"D:\thesis\dataset\WildDeepfake\leakage_free_subset"
    r"\wilddeepfake_sequence_disjoint_face_preprocessed.h5"
)

model = tf.keras.models.load_model(
    MODEL_PATH,
    compile=False
)

# Load only the test split
with h5py.File(H5_PATH, "r") as h5f:
    test_images = h5f["test_images"][:]
    test_labels = h5f["test_labels"][:]

print("Model loaded")
print("Test images:", test_images.shape)
print("Test labels:", test_labels.shape)

Model loaded
Test images: (18000, 160, 160, 3)
Test labels: (18000,)


In [10]:
# ============================================================
# REPEATED INFERENCE RESOURCE PROFILING
# ============================================================

import os
import gc
import time
import threading
import numpy as np
import pandas as pd
import psutil

from scipy.stats import t

from pynvml import (
    nvmlInit,
    nvmlShutdown,
    nvmlDeviceGetHandleByIndex,
    nvmlDeviceGetMemoryInfo,
    nvmlDeviceGetUtilizationRates,
    nvmlDeviceGetPowerUsage,
    NVMLError
)


BATCH_SIZE = 16
NUMBER_OF_RUNS = 5
SAMPLING_INTERVAL = 0.1     # Sample every 100 milliseconds
COOLDOWN_SECONDS = 5
GPU_INDEX = 0


class ResourceMonitor:
    """
    Continuously measures resource consumption during one inference run.

    RAM:
        Current Python process resident memory (RSS).

    GPU memory:
        Total device memory currently used, with the pre-inference
        baseline subtracted for incremental measurements.

    CPU:
        Current Python-process CPU utilization normalized by the
        number of logical CPU cores.

    Power:
        NVIDIA GPU board power reported by NVML.
    """

    def __init__(
        self,
        interval=0.1,
        gpu_index=0
    ):
        self.interval = interval
        self.process = psutil.Process(os.getpid())

        self.logical_cpu_count = (
            psutil.cpu_count(logical=True) or 1
        )

        self.stop_event = threading.Event()
        self.samples = []
        self.thread = None

        nvmlInit()
        self.gpu_handle = nvmlDeviceGetHandleByIndex(
            gpu_index
        )

    def _read_gpu_power(self):
        try:
            return (
                nvmlDeviceGetPowerUsage(
                    self.gpu_handle
                ) / 1000.0
            )
        except NVMLError:
            return np.nan

    def _collect_sample(self):
        timestamp = time.perf_counter()

        # Process RAM: resident set size
        ram_mb = (
            self.process.memory_info().rss
            / (1024 ** 2)
        )

        # Process CPU percentage can exceed 100% on multicore systems.
        # Normalize it to an approximate 0–100% scale.
        process_cpu_raw = self.process.cpu_percent(
            interval=None
        )

        process_cpu_normalized = (
            process_cpu_raw
            / self.logical_cpu_count
        )

        gpu_memory = nvmlDeviceGetMemoryInfo(
            self.gpu_handle
        )

        gpu_memory_used_mb = (
            gpu_memory.used
            / (1024 ** 2)
        )

        gpu_utilization = nvmlDeviceGetUtilizationRates(
            self.gpu_handle
        ).gpu

        gpu_power_w = self._read_gpu_power()

        self.samples.append({
            "timestamp": timestamp,
            "ram_mb": ram_mb,
            "cpu_percent": process_cpu_normalized,
            "gpu_memory_mb": gpu_memory_used_mb,
            "gpu_utilization_percent": gpu_utilization,
            "gpu_power_w": gpu_power_w
        })

    def _sampling_loop(self):
        while not self.stop_event.is_set():
            try:
                self._collect_sample()
            except Exception as error:
                print("Monitoring warning:", error)

            self.stop_event.wait(self.interval)

    def start(self):
        # Initialize CPU counters
        self.process.cpu_percent(interval=None)

        # First sample is the pre-inference baseline
        self._collect_sample()

        self.thread = threading.Thread(
            target=self._sampling_loop,
            daemon=True
        )

        self.thread.start()

    def stop(
        self,
        elapsed_seconds,
        number_of_images
    ):
        self.stop_event.set()

        if self.thread is not None:
            self.thread.join()

        # Capture one final sample
        try:
            self._collect_sample()
        except Exception:
            pass

        nvmlShutdown()

        data = pd.DataFrame(self.samples)

        baseline_ram = data["ram_mb"].iloc[0]
        baseline_gpu_memory = data[
            "gpu_memory_mb"
        ].iloc[0]

        peak_ram = data["ram_mb"].max()
        average_ram = data["ram_mb"].mean()

        peak_gpu_memory = data[
            "gpu_memory_mb"
        ].max()

        average_gpu_memory = data[
            "gpu_memory_mb"
        ].mean()

        peak_incremental_ram = max(
            0.0,
            peak_ram - baseline_ram
        )

        average_incremental_ram = max(
            0.0,
            average_ram - baseline_ram
        )

        peak_incremental_gpu_memory = max(
            0.0,
            peak_gpu_memory - baseline_gpu_memory
        )

        average_incremental_gpu_memory = max(
            0.0,
            average_gpu_memory - baseline_gpu_memory
        )

        # Estimate GPU energy using power integration
        valid_power = data.dropna(
            subset=["gpu_power_w"]
        )

        if len(valid_power) >= 2:
            relative_times = (
                valid_power["timestamp"].to_numpy()
                - valid_power["timestamp"].iloc[0]
            )

            energy_joules = np.trapz(
                valid_power["gpu_power_w"].to_numpy(),
                relative_times
            )

            energy_wh = energy_joules / 3600.0
        else:
            energy_wh = np.nan

        return {
            "elapsed_time_s": elapsed_seconds,

            "latency_ms_per_image": (
                elapsed_seconds
                / number_of_images
                * 1000.0
            ),

            "throughput_images_per_s": (
                number_of_images
                / elapsed_seconds
            ),

            "average_cpu_percent": (
                data["cpu_percent"].mean()
            ),

            "peak_cpu_percent": (
                data["cpu_percent"].max()
            ),

            "baseline_ram_mb": baseline_ram,
            "average_ram_mb": average_ram,
            "peak_ram_mb": peak_ram,

            "average_incremental_ram_mb": (
                average_incremental_ram
            ),

            "peak_incremental_ram_mb": (
                peak_incremental_ram
            ),

            "baseline_gpu_memory_mb": (
                baseline_gpu_memory
            ),

            "average_gpu_memory_mb": (
                average_gpu_memory
            ),

            "peak_gpu_memory_mb": (
                peak_gpu_memory
            ),

            "average_incremental_gpu_memory_mb": (
                average_incremental_gpu_memory
            ),

            "peak_incremental_gpu_memory_mb": (
                peak_incremental_gpu_memory
            ),

            "average_gpu_utilization_percent": (
                data["gpu_utilization_percent"].mean()
            ),

            "peak_gpu_utilization_percent": (
                data["gpu_utilization_percent"].max()
            ),

            "average_gpu_power_w": (
                data["gpu_power_w"].mean()
            ),

            "peak_gpu_power_w": (
                data["gpu_power_w"].max()
            ),

            "gpu_energy_wh": energy_wh
        }


# ------------------------------------------------------------
# GPU/model warm-up
# ------------------------------------------------------------

warmup_count = min(
    len(test_images),
    BATCH_SIZE * 3
)

print("Performing warm-up inference...")

_ = model.predict(
    test_images[:warmup_count],
    batch_size=BATCH_SIZE,
    verbose=0
)

print("Warm-up completed.")


# ------------------------------------------------------------
# Five repeated inference runs
# ------------------------------------------------------------

run_results = []

for run_number in range(
    1,
    NUMBER_OF_RUNS + 1
):
    print(
        f"\nStarting resource run "
        f"{run_number}/{NUMBER_OF_RUNS}"
    )

    gc.collect()
    time.sleep(COOLDOWN_SECONDS)

    monitor = ResourceMonitor(
        interval=SAMPLING_INTERVAL,
        gpu_index=GPU_INDEX
    )

    monitor.start()

    start_time = time.perf_counter()

    predictions = model.predict(
        test_images,
        batch_size=BATCH_SIZE,
        verbose=0
    )

    elapsed_time = (
        time.perf_counter()
        - start_time
    )

    # Access the result to ensure it is materialized
    _ = float(predictions[-1].reshape(-1)[0])

    run_summary = monitor.stop(
        elapsed_seconds=elapsed_time,
        number_of_images=len(test_images)
    )

    run_summary["run"] = run_number
    run_results.append(run_summary)

    print(
        f"Run {run_number}: "
        f"{elapsed_time:.2f} seconds, "
        f"{run_summary['latency_ms_per_image']:.4f} ms/image"
    )

    del predictions


results_df = pd.DataFrame(run_results)

print("\nIndividual runs:")
display(results_df)

C:\Users\aneek\AppData\Local\Temp\ipykernel_23096\2159758100.py:15: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  from pynvml import (


Performing warm-up inference...
Warm-up completed.

Starting resource run 1/5
Run 1: 12.11 seconds, 0.6729 ms/image

Starting resource run 2/5
Run 2: 11.48 seconds, 0.6377 ms/image

Starting resource run 3/5
Run 3: 12.02 seconds, 0.6677 ms/image

Starting resource run 4/5
Run 4: 12.14 seconds, 0.6746 ms/image

Starting resource run 5/5
Run 5: 12.41 seconds, 0.6893 ms/image

Individual runs:


,elapsed_time_s,latency_ms_per_image,throughput_images_per_s,average_cpu_percent,peak_cpu_percent,baseline_ram_mb,average_ram_mb,peak_ram_mb,average_incremental_ram_mb,peak_incremental_ram_mb,...,average_gpu_memory_mb,peak_gpu_memory_mb,average_incremental_gpu_memory_mb,peak_incremental_gpu_memory_mb,average_gpu_utilization_percent,peak_gpu_utilization_percent,average_gpu_power_w,peak_gpu_power_w,gpu_energy_wh,run
0,12.112208,0.672900,1486.103888,3.404109,4.928125,8507.378906,9831.319372,9866.457031,1323.940466,1359.078125,...,7154.714844,7154.714844,0.000000,0.000000,60.657407,100,37.782093,40.366,0.125202,1
1,11.479385,0.637744,1568.028239,3.373542,4.928125,8512.492188,9834.414435,9870.976562,1321.922247,1358.484375,...,7156.938356,7157.382812,2.223512,2.667969,63.866667,84,38.296705,44.150,0.121617,2
2,12.018753,0.667709,1497.659483,3.423814,5.375000,8512.910156,9832.519531,9870.871094,1319.609375,1357.960938,...,7157.382812,7157.382812,0.000000,0.000000,60.629630,100,37.382481,42.844,0.123376,3
3,12.142892,0.674605,1482.348662,3.448977,7.168750,8512.921875,9831.466016,9870.890625,1318.544141,1357.968750,...,7157.382812,7157.382812,0.000000,0.000000,59.381818,100,37.744100,47.935,0.125667,4
4,12.407259,0.689292,1450.763622,3.334572,4.928125,8514.687500,9838.639956,9872.894531,1323.952456,1358.207031,...,7125.679934,7162.027344,0.000000,7.312500,59.657658,100,38.437054,46.238,0.131251,5


In [11]:
# ============================================================RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# ============================================================

metrics_to_report = [
    "elapsed_time_s",
    "latency_ms_per_image",
    "throughput_images_per_s",

    "average_cpu_percent",
    "peak_cpu_percent",

    "average_ram_mb",
    "peak_ram_mb",
    "average_incremental_ram_mb",
    "peak_incremental_ram_mb",

    "average_gpu_memory_mb",
    "peak_gpu_memory_mb",
    "average_incremental_gpu_memory_mb",
    "peak_incremental_gpu_memory_mb",

    "average_gpu_utilization_percent",
    "peak_gpu_utilization_percent",

    "average_gpu_power_w",
    "peak_gpu_power_w",
    "gpu_energy_wh"
]


summary_rows = []

number_of_runs = len(results_df)

for metric in metrics_to_report:
    values = results_df[metric].dropna()

    mean_value = values.mean()
    standard_deviation = values.std(ddof=1)

    if len(values) > 1:
        critical_t = t.ppf(
            0.975,
            df=len(values) - 1
        )

        confidence_half_width = (
            critical_t
            * standard_deviation
            / np.sqrt(len(values))
        )
    else:
        confidence_half_width = np.nan

    summary_rows.append({
        "Metric": metric,
        "Mean": mean_value,
        "Standard Deviation": standard_deviation,
        "95% CI Lower": (
            mean_value - confidence_half_width
        ),
        "95% CI Upper": (
            mean_value + confidence_half_width
        )
    })


resource_summary = pd.DataFrame(summary_rows)

print("\n" + "=" * 90)
print("EfficientNetV2_B0 RESOURCE CONSUMPTION — FIVE INFERENCE RUNS")
print("=" * 90)

display(resource_summary)


print("\nMain values for the manuscript")
print("-" * 90)

for metric in [
    "latency_ms_per_image",
    "peak_ram_mb",
    "peak_gpu_memory_mb",
    "average_gpu_utilization_percent",
    "average_gpu_power_w"
]:
    row = resource_summary[
        resource_summary["Metric"] == metric
    ].iloc[0]

    print(
        f"{metric}: "
        f"{row['Mean']:.3f} ± "
        f"{row['Standard Deviation']:.3f} "
        f"(95% CI: "
        f"{row['95% CI Lower']:.3f}–"
        f"{row['95% CI Upper']:.3f})"
    )


EfficientNetV2_B0 RESOURCE CONSUMPTION — FIVE INFERENCE RUNS


,Metric,Mean,Standard Deviation,95% CI Lower,95% CI Upper
0,elapsed_time_s,12.032100,0.340994,11.608699,12.455500
1,latency_ms_per_image,0.668450,0.018944,0.644928,0.691972
2,throughput_images_per_s,1496.980779,43.349260,1443.155563,1550.805994
3,average_cpu_percent,3.397003,0.044493,3.341757,3.452249
4,peak_cpu_percent,5.465625,0.971541,4.259298,6.671952
5,average_ram_mb,9833.671862,3.039787,9829.897468,9837.446255
6,peak_ram_mb,9870.417969,2.375034,9867.468975,9873.366963
7,average_incremental_ram_mb,1321.593737,2.470645,1318.526026,1324.661448
8,peak_incremental_ram_mb,1358.339844,0.464991,1357.762481,1358.917207
9,average_gpu_memory_mb,7150.419752,13.874140,7133.192732,7167.646772



Main values for the manuscript
------------------------------------------------------------------------------------------
latency_ms_per_image: 0.668 ± 0.019 (95% CI: 0.645–0.692)
peak_ram_mb: 9870.418 ± 2.375 (95% CI: 9867.469–9873.367)
peak_gpu_memory_mb: 7157.778 ± 2.641 (95% CI: 7154.498–7161.058)
average_gpu_utilization_percent: 60.839 ± 1.786 (95% CI: 58.621–63.057)
average_gpu_power_w: 37.928 ± 0.432 (95% CI: 37.392–38.465)


genralization

In [14]:
#celeb on wilddeepfake
test_results = model.evaluate(
    test_celeb,
    test_labels,
    batch_size=16,
    verbose=1,
    return_dict=True
)

print("\nTest results of wild deepfake dataset on Celeb-DF(V2) (EfficientNetB0):")
for metric_name, metric_value in test_results.items():
    print(f"{metric_name}: {metric_value:.4f}")


373/373 [==============================] - 5s 12ms/step - loss: 0.5230 - accuracy: 0.8448 - precision: 0.9266 - recall: 0.8996 - roc_auc: 0.7113 - pr_auc: 0.9504

Test results of wild deepfake dataset on Celeb-DF(V2) (EfficientNetB0):
loss: 0.5230
accuracy: 0.8448
precision: 0.9266
recall: 0.8996
roc_auc: 0.7113
pr_auc: 0.9504


In [16]:
#dfc on wilddeepfake

test_results = model.evaluate(
    test_hog,
    test_labels,
    batch_size=16,
    verbose=1,
    return_dict=True
)

print("\nTest results of wild deepfake dataset on DFC (EfficientNetB0):")
for metric_name, metric_value in test_results.items():
    print(f"{metric_name}: {metric_value:.4f}")

188/188 [==============================] - 4s 14ms/step - loss: 1.5756 - accuracy: 0.5187 - precision: 0.5161 - recall: 0.6000 - roc_auc: 0.5210 - pr_auc: 0.5060

Test results of wild deepfake dataset on DFC (EfficientNetB0):
loss: 1.5756
accuracy: 0.5187
precision: 0.5161
recall: 0.6000
roc_auc: 0.5210
pr_auc: 0.5060


In [18]:
#ff++ on wilddeepfake

test_results = model.evaluate(
    test_ff,
    test_ff_labels,
    batch_size=16,
    verbose=1,
    return_dict=True
)

print("\nTest results of wild deepfake dataset on FF++ (EfficientNetB0):")
for metric_name, metric_value in test_results.items():
    print(f"{metric_name}: {metric_value:.4f}")

155/155 [==============================] - 2s 12ms/step - loss: 1.8733 - accuracy: 0.4945 - precision: 0.4417 - recall: 0.8089 - roc_auc: 0.5792 - pr_auc: 0.4763

Test results of wild deepfake dataset on FF++ (EfficientNetB0):
loss: 1.8733
accuracy: 0.4945
precision: 0.4417
recall: 0.8089
roc_auc: 0.5792
pr_auc: 0.4763


In [13]:
#celeb on wilddeepfake
test_results = model.evaluate(
    test_celeb,
    test_labels,
    batch_size=16,
    verbose=1,
    return_dict=True
)

print("\nTest results of wild deepfake dataset on Celeb-DF(V2) (EfficientNetV2_B0):")
for metric_name, metric_value in test_results.items():
    print(f"{metric_name}: {metric_value:.4f}")


373/373 [==============================] - 5s 11ms/step - loss: 3.0914 - accuracy: 0.3751 - precision: 0.9083 - recall: 0.3436 - roc_auc: 0.5133 - pr_auc: 0.9072

Test results of wild deepfake dataset on Celeb-DF(V2) (EfficientNetV2_B0):
loss: 3.0914
accuracy: 0.3751
precision: 0.9083
recall: 0.3436
roc_auc: 0.5133
pr_auc: 0.9072


In [15]:
#dfc on wilddeepfake

test_results = model.evaluate(
    test_hog,
    test_labels,
    batch_size=16,
    verbose=1,
    return_dict=True
)

print("\nTest results of wild deepfake dataset on DFC (EfficientNetV2_B0):")
for metric_name, metric_value in test_results.items():
    print(f"{metric_name}: {metric_value:.4f}")

188/188 [==============================] - 3s 13ms/step - loss: 2.4619 - accuracy: 0.4527 - precision: 0.4427 - recall: 0.3660 - roc_auc: 0.4355 - pr_auc: 0.4553

Test results of wild deepfake dataset on DFC (EfficientNetV2_B0):
loss: 2.4619
accuracy: 0.4527
precision: 0.4427
recall: 0.3660
roc_auc: 0.4355
pr_auc: 0.4553


In [17]:
#ff++ on wilddeepfake

test_results = model.evaluate(
    test_ff,
    test_ff_labels,
    batch_size=16,
    verbose=1,
    return_dict=True
)

print("\nTest results of wild deepfake dataset on FF++ (EfficientNetV2_B0):")
for metric_name, metric_value in test_results.items():
    print(f"{metric_name}: {metric_value:.4f}")

155/155 [==============================] - 2s 12ms/step - loss: 2.5295 - accuracy: 0.5402 - precision: 0.4193 - recall: 0.2696 - roc_auc: 0.4925 - pr_auc: 0.4166

Test results of wild deepfake dataset on FF++ (EfficientNetV2_B0):
loss: 2.5295
accuracy: 0.5402
precision: 0.4193
recall: 0.2696
roc_auc: 0.4925
pr_auc: 0.4166


# Celeb

In [13]:
import os
import cv2
import numpy as np

SAVE_ROOT = r'D:\thesis\celeb_processed'

def load_split(split_name, class_name):
    """Reload saved frames, grouped by video."""
    base = os.path.join(SAVE_ROOT, split_name, class_name)
    nested, ids = [], []
    for vid_id in sorted(os.listdir(base)):
        vid_dir = os.path.join(base, vid_id)
        frames = [cv2.imread(os.path.join(vid_dir, f))
                  for f in sorted(os.listdir(vid_dir))]
        if frames:
            nested.append(frames)
            ids.append(vid_id)
    return nested, ids

# Reload ALL six splits
print("Loading frames...")
real_train_final,  real_train_ids  = load_split('train', 'real')
synth_train_final, synth_train_ids = load_split('train', 'fake')
real_val_final,    real_val_ids    = load_split('val',   'real')
synth_val_final,   synth_val_ids   = load_split('val',   'fake')
real_test_final,   real_test_ids   = load_split('test',  'real')
synth_test_final,  synth_test_ids  = load_split('test',  'fake')

print("✅ All frames reloaded")
print("Train -> real videos:", len(real_train_final), " fake videos:", len(synth_train_final))
print("Val   -> real videos:", len(real_val_final),   " fake videos:", len(synth_val_final))
print("Test  -> real videos:", len(real_test_final),  " fake videos:", len(synth_test_final))
print("Example frame shape:", np.shape(real_train_final[0][0]))  # expect (160,160,3)
import numpy as np


def combine_split(real_videos, fake_videos):
    """
    Flatten video-grouped frames into one image array and create labels.

    real_videos: list of videos, where each video is a list of frames
    fake_videos: list of videos, where each video is a list of frames

    Returns
    -------
    images : NumPy array with shape (N, 160, 160, 3)
    labels : NumPy array with shape (N,)
             0 = real, 1 = fake
    """

    # Flatten frames from all real videos
    real_frames = [
        frame
        for video_frames in real_videos
        for frame in video_frames
        if frame is not None
    ]

    # Flatten frames from all fake videos
    fake_frames = [
        frame
        for video_frames in fake_videos
        for frame in video_frames
        if frame is not None
    ]

    if len(real_frames) == 0:
        raise ValueError("No real frames were found.")

    if len(fake_frames) == 0:
        raise ValueError("No fake frames were found.")

    # Convert to NumPy arrays
    real_frames = np.stack(real_frames).astype(np.uint8)
    fake_frames = np.stack(fake_frames).astype(np.uint8)

    # Combine images
    images = np.concatenate(
        [real_frames, fake_frames],
        axis=0
    )

    # Create labels
    real_labels = np.zeros(
        len(real_frames),
        dtype=np.uint8
    )

    fake_labels = np.ones(
        len(fake_frames),
        dtype=np.uint8
    )

    labels = np.concatenate(
        [real_labels, fake_labels],
        axis=0
    )

    return images, labels
# Training set
train_celeb, train_labels = combine_split(
    real_train_final,
    synth_train_final
)

# Validation set
val_celeb, val_labels = combine_split(
    real_val_final,
    synth_val_final
)

# Testing set
test_celeb, test_labels = combine_split(
    real_test_final,
    synth_test_final
)
print("\nTRAIN")
print("Images:", train_celeb.shape)
print("Labels:", train_labels.shape)
print("Real:", np.sum(train_labels == 0))
print("Fake:", np.sum(train_labels == 1))

print("\nVALIDATION")
print("Images:", val_celeb.shape)
print("Labels:", val_labels.shape)
print("Real:", np.sum(val_labels == 0))
print("Fake:", np.sum(val_labels == 1))

print("\nTEST")
print("Images:", test_celeb.shape)
print("Labels:", test_labels.shape)
print("Real:", np.sum(test_labels == 0))
print("Fake:", np.sum(test_labels == 1))

print("\nData types")
print("Train images:", train_celeb.dtype)
print("Train labels:", train_labels.dtype)

Loading frames...
✅ All frames reloaded
Train -> real videos: 354  fake videos: 3383
Val   -> real videos: 59  fake videos: 563
Test  -> real videos: 177  fake videos: 1693
Example frame shape: (160, 160, 3)

TRAIN
Images: (11899, 160, 160, 3)
Labels: (11899,)
Real: 1142
Fake: 10757

VALIDATION
Images: (1969, 160, 160, 3)
Labels: (1969,)
Real: 182
Fake: 1787

TEST
Images: (5961, 160, 160, 3)
Labels: (5961,)
Real: 571
Fake: 5390

Data types
Train images: uint8
Train labels: uint8


In [5]:
# Train model
history = model.fit(
    train_celeb, train_labels,
    validation_data=(val_celeb, val_labels),
    epochs=10,
    batch_size=16,
    verbose=1
)

Epoch 1/10
744/744 [==============================] - 37s 40ms/step - loss: 0.3295 - accuracy: 0.9034 - precision: 0.9042 - recall: 0.9991 - roc_auc: 0.5086 - pr_auc: 0.9036 - val_loss: 0.3080 - val_accuracy: 0.9076 - val_precision: 0.9076 - val_recall: 1.0000 - val_roc_auc: 0.6030 - val_pr_auc: 0.9337
Epoch 2/10
744/744 [==============================] - 30s 41ms/step - loss: 0.3243 - accuracy: 0.9040 - precision: 0.9040 - recall: 1.0000 - roc_auc: 0.5117 - pr_auc: 0.9073 - val_loss: 0.3066 - val_accuracy: 0.9076 - val_precision: 0.9076 - val_recall: 1.0000 - val_roc_auc: 0.5698 - val_pr_auc: 0.9267
Epoch 3/10
744/744 [==============================] - 31s 41ms/step - loss: 0.3205 - accuracy: 0.9040 - precision: 0.9040 - recall: 1.0000 - roc_auc: 0.5329 - pr_auc: 0.9134 - val_loss: 0.3133 - val_accuracy: 0.9076 - val_precision: 0.9076 - val_recall: 1.0000 - val_roc_auc: 0.5974 - val_pr_auc: 0.9337
Epoch 4/10
744/744 [==============================] - 31s 41ms/step - loss: 0.3179 - acc

In [8]:
print("=== DATA LOADING ===")
start = monitor.get_stats()

=== DATA LOADING ===


In [9]:
# ============================================================
# COMPLETE TEST EVALUATION
# Labels: 0 = Real, 1 = Fake
# ============================================================

import numpy as np

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    roc_curve,
    auc,
    confusion_matrix,
    classification_report,
    matthews_corrcoef
)

BATCH_SIZE = 16
DECISION_THRESHOLD = 0.50


# ------------------------------------------------------------
# 1. Test loss
# ------------------------------------------------------------

evaluation = model.evaluate(
    test_celeb,
    test_labels,
    batch_size=BATCH_SIZE,
    verbose=1,
    return_dict=True
)

test_loss = evaluation["loss"]


# ------------------------------------------------------------
# 2. Prediction probabilities and binary predictions
# ------------------------------------------------------------

test_probabilities = model.predict(
    test_celeb,
    batch_size=BATCH_SIZE,
    verbose=1
).reshape(-1)

y_true = np.asarray(test_labels).reshape(-1).astype(np.uint8)

y_pred = (
    test_probabilities >= DECISION_THRESHOLD
).astype(np.uint8)


# ------------------------------------------------------------
# 3. Confusion matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1]
)

tn, fp, fn, tp = cm.ravel()


# ------------------------------------------------------------
# 4. Threshold-dependent metrics
# ------------------------------------------------------------

accuracy = accuracy_score(y_true, y_pred)

balanced_accuracy = balanced_accuracy_score(
    y_true,
    y_pred
)

precision = precision_score(
    y_true,
    y_pred,
    pos_label=1,
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    pos_label=1,
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    pos_label=1,
    zero_division=0
)

specificity = (
    tn / (tn + fp)
    if (tn + fp) > 0
    else 0.0
)

false_positive_rate_at_05 = (
    fp / (fp + tn)
    if (fp + tn) > 0
    else 0.0
)

false_negative_rate_at_05 = (
    fn / (fn + tp)
    if (fn + tp) > 0
    else 0.0
)

mcc = matthews_corrcoef(
    y_true,
    y_pred
)


# ------------------------------------------------------------
# 5. ROC-AUC
# ------------------------------------------------------------

roc_auc = roc_auc_score(
    y_true,
    test_probabilities
)

fpr, tpr, roc_thresholds = roc_curve(
    y_true,
    test_probabilities,
    pos_label=1
)


# ------------------------------------------------------------
# 6. PR-AUC and Average Precision
# ------------------------------------------------------------

pr_precision, pr_recall, _ = precision_recall_curve(
    y_true,
    test_probabilities,
    pos_label=1
)

# Reverse because recall is normally returned in descending order
pr_auc = auc(
    pr_recall[::-1],
    pr_precision[::-1]
)

average_precision = average_precision_score(
    y_true,
    test_probabilities
)


# ------------------------------------------------------------
# 7. Equal Error Rate
# ------------------------------------------------------------

fnr_curve = 1.0 - tpr

# Remove non-finite thresholds such as infinity
valid_indices = np.where(
    np.isfinite(roc_thresholds)
)[0]

eer_index = valid_indices[
    np.argmin(
        np.abs(
            fpr[valid_indices]
            - fnr_curve[valid_indices]
        )
    )
]

eer = (
    fpr[eer_index]
    + fnr_curve[eer_index]
) / 2.0

eer_threshold = roc_thresholds[eer_index]


# ------------------------------------------------------------
# 8. Display all results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("EfficientNetB0 — COMPLETE TEST-SET EVALUATION")
print("=" * 70)

print(f"Number of test samples:     {len(y_true)}")
print(f"Decision threshold:         {DECISION_THRESHOLD:.4f}")
print(f"Test loss:                  {test_loss:.6f}")

print("\nMain evaluation metrics")
print("-" * 70)
print(f"Accuracy:                   {accuracy:.6f} ({accuracy*100:.2f}%)")
print(f"Precision:                  {precision:.6f} ({precision*100:.2f}%)")
print(f"Recall/Sensitivity:         {recall:.6f} ({recall*100:.2f}%)")
print(f"F1-score:                   {f1:.6f} ({f1*100:.2f}%)")
print(f"ROC-AUC:                    {roc_auc:.6f}")
print(f"PR-AUC:                     {pr_auc:.6f}")
print(f"Average Precision:          {average_precision:.6f}")
print(f"EER:                        {eer:.6f} ({eer*100:.2f}%)")
print(f"EER threshold:              {eer_threshold:.6f}")

print("\nAdditional evaluation metrics")
print("-" * 70)
print(f"Balanced accuracy:          {balanced_accuracy:.6f}")
print(f"Specificity:                {specificity:.6f}")
print(f"Matthews correlation:       {mcc:.6f}")
print(f"False-positive rate @ 0.5:  {false_positive_rate_at_05:.6f}")
print(f"False-negative rate @ 0.5:  {false_negative_rate_at_05:.6f}")

print("\nConfusion matrix")
print("-" * 70)
print("Rows = actual classes; columns = predicted classes")
print("Class order: [Real, Fake]")
print(cm)

print("\nConfusion-matrix values")
print("-" * 70)
print(f"True Negative  — Real predicted as Real: {tn}")
print(f"False Positive — Real predicted as Fake: {fp}")
print(f"False Negative — Fake predicted as Real: {fn}")
print(f"True Positive  — Fake predicted as Fake: {tp}")

print("\nClassification report")
print("-" * 70)

print(
    classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=["Real", "Fake"],
        digits=6,
        zero_division=0
    )
)

373/373 [==============================] - 6s 12ms/step

EfficientNetB0 — COMPLETE TEST-SET EVALUATION
Number of test samples:     5961
Decision threshold:         0.5000
Test loss:                  0.323469

Main evaluation metrics
----------------------------------------------------------------------
Accuracy:                   0.903707 (90.37%)
Precision:                  0.904162 (90.42%)
Recall/Sensitivity:         0.999443 (99.94%)
F1-score:                   0.949418 (94.94%)
ROC-AUC:                    0.570589
PR-AUC:                     0.923492
Average Precision:          0.923545
EER:                        0.452079 (45.21%)
EER threshold:              0.870798

Additional evaluation metrics
----------------------------------------------------------------------
Balanced accuracy:          0.499722
Specificity:                0.000000
Matthews correlation:       -0.007304
False-positive rate @ 0.5:  1.000000
False-negative rate @ 0.5:  0.000557

Confusion matrix
------------

In [7]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"RAM Used: {end['ram_mb'] - start['ram_mb']:.1f} MB")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 14.0%
RAM Used: 425.8 MB
Time Usage: 6.8 s
GPU Memory Used: 1538.5 MB
Power Consumption: 93W


save the model

In [ ]:
MODEL_PATH = (
    r"D:\thesis\results"
    r"\EfficientNetB0_celeb_10epochs.h5"
)

model.save(
    MODEL_PATH,
    include_optimizer=True
)

print("Model saved successfully:")
print(MODEL_PATH)
# After training the model
model.save('EfficientNetB03_160_celeb.h5')  # Saves the entire model to a file

load the model

In [ ]:
import h5py
import tensorflow as tf

MODEL_PATH = ( r"D:\thesis\results" r"\EfficientNetB0_celeb_10epochs.h5")


model = tf.keras.models.load_model(
    MODEL_PATH,
    compile=False
)



In [ ]:
import h5py
import tensorflow as tf

MODEL_PATH = ( r"Mesonet_160_celeb.h5")


In [9]:
# ============================================================
# REPEATED INFERENCE RESOURCE PROFILING
# ============================================================

import os
import gc
import time
import threading
import numpy as np
import pandas as pd
import psutil

from scipy.stats import t

from pynvml import (
    nvmlInit,
    nvmlShutdown,
    nvmlDeviceGetHandleByIndex,
    nvmlDeviceGetMemoryInfo,
    nvmlDeviceGetUtilizationRates,
    nvmlDeviceGetPowerUsage,
    NVMLError
)


BATCH_SIZE = 16
NUMBER_OF_RUNS = 5
SAMPLING_INTERVAL = 0.1     # Sample every 100 milliseconds
COOLDOWN_SECONDS = 5
GPU_INDEX = 0


class ResourceMonitor:
    """
    Continuously measures resource consumption during one inference run.

    RAM:
        Current Python process resident memory (RSS).

    GPU memory:
        Total device memory currently used, with the pre-inference
        baseline subtracted for incremental measurements.

    CPU:
        Current Python-process CPU utilization normalized by the
        number of logical CPU cores.

    Power:
        NVIDIA GPU board power reported by NVML.
    """

    def __init__(
        self,
        interval=0.1,
        gpu_index=0
    ):
        self.interval = interval
        self.process = psutil.Process(os.getpid())

        self.logical_cpu_count = (
            psutil.cpu_count(logical=True) or 1
        )

        self.stop_event = threading.Event()
        self.samples = []
        self.thread = None

        nvmlInit()
        self.gpu_handle = nvmlDeviceGetHandleByIndex(
            gpu_index
        )

    def _read_gpu_power(self):
        try:
            return (
                nvmlDeviceGetPowerUsage(
                    self.gpu_handle
                ) / 1000.0
            )
        except NVMLError:
            return np.nan

    def _collect_sample(self):
        timestamp = time.perf_counter()

        # Process RAM: resident set size
        ram_mb = (
            self.process.memory_info().rss
            / (1024 ** 2)
        )

        # Process CPU percentage can exceed 100% on multicore systems.
        # Normalize it to an approximate 0–100% scale.
        process_cpu_raw = self.process.cpu_percent(
            interval=None
        )

        process_cpu_normalized = (
            process_cpu_raw
            / self.logical_cpu_count
        )

        gpu_memory = nvmlDeviceGetMemoryInfo(
            self.gpu_handle
        )

        gpu_memory_used_mb = (
            gpu_memory.used
            / (1024 ** 2)
        )

        gpu_utilization = nvmlDeviceGetUtilizationRates(
            self.gpu_handle
        ).gpu

        gpu_power_w = self._read_gpu_power()

        self.samples.append({
            "timestamp": timestamp,
            "ram_mb": ram_mb,
            "cpu_percent": process_cpu_normalized,
            "gpu_memory_mb": gpu_memory_used_mb,
            "gpu_utilization_percent": gpu_utilization,
            "gpu_power_w": gpu_power_w
        })

    def _sampling_loop(self):
        while not self.stop_event.is_set():
            try:
                self._collect_sample()
            except Exception as error:
                print("Monitoring warning:", error)

            self.stop_event.wait(self.interval)

    def start(self):
        # Initialize CPU counters
        self.process.cpu_percent(interval=None)

        # First sample is the pre-inference baseline
        self._collect_sample()

        self.thread = threading.Thread(
            target=self._sampling_loop,
            daemon=True
        )

        self.thread.start()

    def stop(
        self,
        elapsed_seconds,
        number_of_images
    ):
        self.stop_event.set()

        if self.thread is not None:
            self.thread.join()

        # Capture one final sample
        try:
            self._collect_sample()
        except Exception:
            pass

        nvmlShutdown()

        data = pd.DataFrame(self.samples)

        baseline_ram = data["ram_mb"].iloc[0]
        baseline_gpu_memory = data[
            "gpu_memory_mb"
        ].iloc[0]

        peak_ram = data["ram_mb"].max()
        average_ram = data["ram_mb"].mean()

        peak_gpu_memory = data[
            "gpu_memory_mb"
        ].max()

        average_gpu_memory = data[
            "gpu_memory_mb"
        ].mean()

        peak_incremental_ram = max(
            0.0,
            peak_ram - baseline_ram
        )

        average_incremental_ram = max(
            0.0,
            average_ram - baseline_ram
        )

        peak_incremental_gpu_memory = max(
            0.0,
            peak_gpu_memory - baseline_gpu_memory
        )

        average_incremental_gpu_memory = max(
            0.0,
            average_gpu_memory - baseline_gpu_memory
        )

        # Estimate GPU energy using power integration
        valid_power = data.dropna(
            subset=["gpu_power_w"]
        )

        if len(valid_power) >= 2:
            relative_times = (
                valid_power["timestamp"].to_numpy()
                - valid_power["timestamp"].iloc[0]
            )

            energy_joules = np.trapz(
                valid_power["gpu_power_w"].to_numpy(),
                relative_times
            )

            energy_wh = energy_joules / 3600.0
        else:
            energy_wh = np.nan

        return {
            "elapsed_time_s": elapsed_seconds,

            "latency_ms_per_image": (
                elapsed_seconds
                / number_of_images
                * 1000.0
            ),

            "throughput_images_per_s": (
                number_of_images
                / elapsed_seconds
            ),

            "average_cpu_percent": (
                data["cpu_percent"].mean()
            ),

            "peak_cpu_percent": (
                data["cpu_percent"].max()
            ),

            "baseline_ram_mb": baseline_ram,
            "average_ram_mb": average_ram,
            "peak_ram_mb": peak_ram,

            "average_incremental_ram_mb": (
                average_incremental_ram
            ),

            "peak_incremental_ram_mb": (
                peak_incremental_ram
            ),

            "baseline_gpu_memory_mb": (
                baseline_gpu_memory
            ),

            "average_gpu_memory_mb": (
                average_gpu_memory
            ),

            "peak_gpu_memory_mb": (
                peak_gpu_memory
            ),

            "average_incremental_gpu_memory_mb": (
                average_incremental_gpu_memory
            ),

            "peak_incremental_gpu_memory_mb": (
                peak_incremental_gpu_memory
            ),

            "average_gpu_utilization_percent": (
                data["gpu_utilization_percent"].mean()
            ),

            "peak_gpu_utilization_percent": (
                data["gpu_utilization_percent"].max()
            ),

            "average_gpu_power_w": (
                data["gpu_power_w"].mean()
            ),

            "peak_gpu_power_w": (
                data["gpu_power_w"].max()
            ),

            "gpu_energy_wh": energy_wh
        }


# ------------------------------------------------------------
# GPU/model warm-up
# ------------------------------------------------------------

warmup_count = min(
    len(test_celeb),
    BATCH_SIZE * 3
)

print("Performing warm-up inference...")

_ = model.predict(
    test_celeb[:warmup_count],
    batch_size=BATCH_SIZE,
    verbose=0
)

print("Warm-up completed.")


# ------------------------------------------------------------
# Five repeated inference runs
# ------------------------------------------------------------

run_results = []

for run_number in range(
    1,
    NUMBER_OF_RUNS + 1
):
    print(
        f"\nStarting resource run "
        f"{run_number}/{NUMBER_OF_RUNS}"
    )

    gc.collect()
    time.sleep(COOLDOWN_SECONDS)

    monitor = ResourceMonitor(
        interval=SAMPLING_INTERVAL,
        gpu_index=GPU_INDEX
    )

    monitor.start()

    start_time = time.perf_counter()

    predictions = model.predict(
        test_celeb,
        batch_size=BATCH_SIZE,
        verbose=0
    )

    elapsed_time = (
        time.perf_counter()
        - start_time
    )

    # Access the result to ensure it is materialized
    _ = float(predictions[-1].reshape(-1)[0])

    run_summary = monitor.stop(
        elapsed_seconds=elapsed_time,
        number_of_images=len(test_celeb)
    )

    run_summary["run"] = run_number
    run_results.append(run_summary)

    print(
        f"Run {run_number}: "
        f"{elapsed_time:.2f} seconds, "
        f"{run_summary['latency_ms_per_image']:.4f} ms/image"
    )

    del predictions


results_df = pd.DataFrame(run_results)

print("\nIndividual runs:")
display(results_df)

C:\Users\aneek\AppData\Local\Temp\ipykernel_33820\3984712684.py:15: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  from pynvml import (


Performing warm-up inference...
Warm-up completed.

Starting resource run 1/5
Run 1: 4.66 seconds, 0.7821 ms/image

Starting resource run 2/5
Run 2: 3.98 seconds, 0.6673 ms/image

Starting resource run 3/5
Run 3: 3.76 seconds, 0.6307 ms/image

Starting resource run 4/5
Run 4: 3.93 seconds, 0.6593 ms/image

Starting resource run 5/5
Run 5: 3.75 seconds, 0.6283 ms/image

Individual runs:


,elapsed_time_s,latency_ms_per_image,throughput_images_per_s,average_cpu_percent,peak_cpu_percent,baseline_ram_mb,average_ram_mb,peak_ram_mb,average_incremental_ram_mb,peak_incremental_ram_mb,...,average_gpu_memory_mb,peak_gpu_memory_mb,average_incremental_gpu_memory_mb,peak_incremental_gpu_memory_mb,average_gpu_utilization_percent,peak_gpu_utilization_percent,average_gpu_power_w,peak_gpu_power_w,gpu_energy_wh,run
0,4.662059,0.782093,1278.619624,3.126308,4.928125,6963.582031,7436.692860,7473.117188,473.110828,509.535156,...,6982.839844,6982.839844,0.0,0.0,65.930233,92,46.747814,71.803,0.059727,1
1,3.977779,0.667301,1498.574846,3.187664,6.271875,7002.898438,7448.949424,7478.472656,446.050987,475.574219,...,6982.839844,6982.839844,0.0,0.0,71.815789,80,46.387237,61.547,0.051932,2
2,3.759571,0.630695,1585.553077,3.002517,5.193750,7004.402344,7448.937500,7479.441406,444.535156,475.039062,...,6982.839844,6982.839844,0.0,0.0,73.833333,85,47.046028,69.321,0.049811,3
3,3.929938,0.659275,1516.817707,3.152196,4.478125,7005.089844,7450.641153,7480.304688,445.551309,475.214844,...,6982.839844,6982.839844,0.0,0.0,74.540541,87,44.800784,61.584,0.050291,4
4,3.745345,0.628308,1591.575676,2.975868,4.478125,7005.121094,7449.904514,7480.238281,444.783420,475.117188,...,6982.839844,6982.839844,0.0,0.0,73.916667,84,47.447861,73.612,0.050282,5


In [10]:
# ============================================================RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# ============================================================

metrics_to_report = [
    "elapsed_time_s",
    "latency_ms_per_image",
    "throughput_images_per_s",

    "average_cpu_percent",
    "peak_cpu_percent",

    "average_ram_mb",
    "peak_ram_mb",
    "average_incremental_ram_mb",
    "peak_incremental_ram_mb",

    "average_gpu_memory_mb",
    "peak_gpu_memory_mb",
    "average_incremental_gpu_memory_mb",
    "peak_incremental_gpu_memory_mb",

    "average_gpu_utilization_percent",
    "peak_gpu_utilization_percent",

    "average_gpu_power_w",
    "peak_gpu_power_w",
    "gpu_energy_wh"
]


summary_rows = []

number_of_runs = len(results_df)

for metric in metrics_to_report:
    values = results_df[metric].dropna()

    mean_value = values.mean()
    standard_deviation = values.std(ddof=1)

    if len(values) > 1:
        critical_t = t.ppf(
            0.975,
            df=len(values) - 1
        )

        confidence_half_width = (
            critical_t
            * standard_deviation
            / np.sqrt(len(values))
        )
    else:
        confidence_half_width = np.nan

    summary_rows.append({
        "Metric": metric,
        "Mean": mean_value,
        "Standard Deviation": standard_deviation,
        "95% CI Lower": (
            mean_value - confidence_half_width
        ),
        "95% CI Upper": (
            mean_value + confidence_half_width
        )
    })


resource_summary = pd.DataFrame(summary_rows)

print("\n" + "=" * 90)
print("EfficientNetB0 RESOURCE CONSUMPTION — FIVE INFERENCE RUNS")
print("=" * 90)

display(resource_summary)


print("\nMain values for the manuscript")
print("-" * 90)

for metric in [
    "latency_ms_per_image",
    "peak_ram_mb",
    "peak_gpu_memory_mb",
    "average_gpu_utilization_percent",
    "average_gpu_power_w"
]:
    row = resource_summary[
        resource_summary["Metric"] == metric
    ].iloc[0]

    print(
        f"{metric}: "
        f"{row['Mean']:.3f} ± "
        f"{row['Standard Deviation']:.3f} "
        f"(95% CI: "
        f"{row['95% CI Lower']:.3f}–"
        f"{row['95% CI Upper']:.3f})"
    )


EfficientNetB0 RESOURCE CONSUMPTION — FIVE INFERENCE RUNS


,Metric,Mean,Standard Deviation,95% CI Lower,95% CI Upper
0,elapsed_time_s,4.014939,0.375920,3.548172,4.481705
1,latency_ms_per_image,0.673534,0.063063,0.595231,0.751838
2,throughput_images_per_s,1494.228186,127.311627,1336.149922,1652.306449
3,average_cpu_percent,3.088911,0.094072,2.972105,3.205717
4,peak_cpu_percent,5.070000,0.738339,4.153231,5.986769
5,average_ram_mb,7447.025090,5.819844,7439.798799,7454.251381
6,peak_ram_mb,7478.314844,2.998698,7474.591468,7482.038220
7,average_incremental_ram_mb,450.806340,12.483211,435.306386,466.306294
8,peak_incremental_ram_mb,482.096094,15.340269,463.048634,501.143554
9,average_gpu_memory_mb,6982.839844,0.000000,6982.839844,6982.839844



Main values for the manuscript
------------------------------------------------------------------------------------------
latency_ms_per_image: 0.674 ± 0.063 (95% CI: 0.595–0.752)
peak_ram_mb: 7478.315 ± 2.999 (95% CI: 7474.591–7482.038)
peak_gpu_memory_mb: 6982.840 ± 0.000 (95% CI: 6982.840–6982.840)
average_gpu_utilization_percent: 72.007 ± 3.548 (95% CI: 67.601–76.413)
average_gpu_power_w: 46.486 ± 1.019 (95% CI: 45.220–47.752)


In [11]:
# ============================================================
# REPEATED INFERENCE RESOURCE PROFILING
# ============================================================

import os
import gc
import time
import threading
import numpy as np
import pandas as pd
import psutil

from scipy.stats import t

from pynvml import (
    nvmlInit,
    nvmlShutdown,
    nvmlDeviceGetHandleByIndex,
    nvmlDeviceGetMemoryInfo,
    nvmlDeviceGetUtilizationRates,
    nvmlDeviceGetPowerUsage,
    NVMLError
)


BATCH_SIZE = 16
NUMBER_OF_RUNS = 5
SAMPLING_INTERVAL = 0.1     # Sample every 100 milliseconds
COOLDOWN_SECONDS = 5
GPU_INDEX = 0


class ResourceMonitor:
    """
    Continuously measures resource consumption during one inference run.

    RAM:
        Current Python process resident memory (RSS).

    GPU memory:
        Total device memory currently used, with the pre-inference
        baseline subtracted for incremental measurements.

    CPU:
        Current Python-process CPU utilization normalized by the
        number of logical CPU cores.

    Power:
        NVIDIA GPU board power reported by NVML.
    """

    def __init__(
        self,
        interval=0.1,
        gpu_index=0
    ):
        self.interval = interval
        self.process = psutil.Process(os.getpid())

        self.logical_cpu_count = (
            psutil.cpu_count(logical=True) or 1
        )

        self.stop_event = threading.Event()
        self.samples = []
        self.thread = None

        nvmlInit()
        self.gpu_handle = nvmlDeviceGetHandleByIndex(
            gpu_index
        )

    def _read_gpu_power(self):
        try:
            return (
                nvmlDeviceGetPowerUsage(
                    self.gpu_handle
                ) / 1000.0
            )
        except NVMLError:
            return np.nan

    def _collect_sample(self):
        timestamp = time.perf_counter()

        # Process RAM: resident set size
        ram_mb = (
            self.process.memory_info().rss
            / (1024 ** 2)
        )

        # Process CPU percentage can exceed 100% on multicore systems.
        # Normalize it to an approximate 0–100% scale.
        process_cpu_raw = self.process.cpu_percent(
            interval=None
        )

        process_cpu_normalized = (
            process_cpu_raw
            / self.logical_cpu_count
        )

        gpu_memory = nvmlDeviceGetMemoryInfo(
            self.gpu_handle
        )

        gpu_memory_used_mb = (
            gpu_memory.used
            / (1024 ** 2)
        )

        gpu_utilization = nvmlDeviceGetUtilizationRates(
            self.gpu_handle
        ).gpu

        gpu_power_w = self._read_gpu_power()

        self.samples.append({
            "timestamp": timestamp,
            "ram_mb": ram_mb,
            "cpu_percent": process_cpu_normalized,
            "gpu_memory_mb": gpu_memory_used_mb,
            "gpu_utilization_percent": gpu_utilization,
            "gpu_power_w": gpu_power_w
        })

    def _sampling_loop(self):
        while not self.stop_event.is_set():
            try:
                self._collect_sample()
            except Exception as error:
                print("Monitoring warning:", error)

            self.stop_event.wait(self.interval)

    def start(self):
        # Initialize CPU counters
        self.process.cpu_percent(interval=None)

        # First sample is the pre-inference baseline
        self._collect_sample()

        self.thread = threading.Thread(
            target=self._sampling_loop,
            daemon=True
        )

        self.thread.start()

    def stop(
        self,
        elapsed_seconds,
        number_of_images
    ):
        self.stop_event.set()

        if self.thread is not None:
            self.thread.join()

        # Capture one final sample
        try:
            self._collect_sample()
        except Exception:
            pass

        nvmlShutdown()

        data = pd.DataFrame(self.samples)

        baseline_ram = data["ram_mb"].iloc[0]
        baseline_gpu_memory = data[
            "gpu_memory_mb"
        ].iloc[0]

        peak_ram = data["ram_mb"].max()
        average_ram = data["ram_mb"].mean()

        peak_gpu_memory = data[
            "gpu_memory_mb"
        ].max()

        average_gpu_memory = data[
            "gpu_memory_mb"
        ].mean()

        peak_incremental_ram = max(
            0.0,
            peak_ram - baseline_ram
        )

        average_incremental_ram = max(
            0.0,
            average_ram - baseline_ram
        )

        peak_incremental_gpu_memory = max(
            0.0,
            peak_gpu_memory - baseline_gpu_memory
        )

        average_incremental_gpu_memory = max(
            0.0,
            average_gpu_memory - baseline_gpu_memory
        )

        # Estimate GPU energy using power integration
        valid_power = data.dropna(
            subset=["gpu_power_w"]
        )

        if len(valid_power) >= 2:
            relative_times = (
                valid_power["timestamp"].to_numpy()
                - valid_power["timestamp"].iloc[0]
            )

            energy_joules = np.trapz(
                valid_power["gpu_power_w"].to_numpy(),
                relative_times
            )

            energy_wh = energy_joules / 3600.0
        else:
            energy_wh = np.nan

        return {
            "elapsed_time_s": elapsed_seconds,

            "latency_ms_per_image": (
                elapsed_seconds
                / number_of_images
                * 1000.0
            ),

            "throughput_images_per_s": (
                number_of_images
                / elapsed_seconds
            ),

            "average_cpu_percent": (
                data["cpu_percent"].mean()
            ),

            "peak_cpu_percent": (
                data["cpu_percent"].max()
            ),

            "baseline_ram_mb": baseline_ram,
            "average_ram_mb": average_ram,
            "peak_ram_mb": peak_ram,

            "average_incremental_ram_mb": (
                average_incremental_ram
            ),

            "peak_incremental_ram_mb": (
                peak_incremental_ram
            ),

            "baseline_gpu_memory_mb": (
                baseline_gpu_memory
            ),

            "average_gpu_memory_mb": (
                average_gpu_memory
            ),

            "peak_gpu_memory_mb": (
                peak_gpu_memory
            ),

            "average_incremental_gpu_memory_mb": (
                average_incremental_gpu_memory
            ),

            "peak_incremental_gpu_memory_mb": (
                peak_incremental_gpu_memory
            ),

            "average_gpu_utilization_percent": (
                data["gpu_utilization_percent"].mean()
            ),

            "peak_gpu_utilization_percent": (
                data["gpu_utilization_percent"].max()
            ),

            "average_gpu_power_w": (
                data["gpu_power_w"].mean()
            ),

            "peak_gpu_power_w": (
                data["gpu_power_w"].max()
            ),

            "gpu_energy_wh": energy_wh
        }


# ------------------------------------------------------------
# GPU/model warm-up
# ------------------------------------------------------------

warmup_count = min(
    len(test_celeb),
    BATCH_SIZE * 3
)

print("Performing warm-up inference...")

_ = model.predict(
    test_celeb[:warmup_count],
    batch_size=BATCH_SIZE,
    verbose=0
)

print("Warm-up completed.")


# ------------------------------------------------------------
# Five repeated inference runs
# ------------------------------------------------------------

run_results = []

for run_number in range(
    1,
    NUMBER_OF_RUNS + 1
):
    print(
        f"\nStarting resource run "
        f"{run_number}/{NUMBER_OF_RUNS}"
    )

    gc.collect()
    time.sleep(COOLDOWN_SECONDS)

    monitor = ResourceMonitor(
        interval=SAMPLING_INTERVAL,
        gpu_index=GPU_INDEX
    )

    monitor.start()

    start_time = time.perf_counter()

    predictions = model.predict(
        test_celeb,
        batch_size=BATCH_SIZE,
        verbose=0
    )

    elapsed_time = (
        time.perf_counter()
        - start_time
    )

    # Access the result to ensure it is materialized
    _ = float(predictions[-1].reshape(-1)[0])

    run_summary = monitor.stop(
        elapsed_seconds=elapsed_time,
        number_of_images=len(test_celeb)
    )

    run_summary["run"] = run_number
    run_results.append(run_summary)

    print(
        f"Run {run_number}: "
        f"{elapsed_time:.2f} seconds, "
        f"{run_summary['latency_ms_per_image']:.4f} ms/image"
    )

    del predictions


results_df = pd.DataFrame(run_results)

print("\nIndividual runs:")
display(results_df)

C:\Users\aneek\AppData\Local\Temp\ipykernel_46960\3984712684.py:15: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  from pynvml import (


Performing warm-up inference...
Warm-up completed.

Starting resource run 1/5
Run 1: 3.64 seconds, 0.6110 ms/image

Starting resource run 2/5
Run 2: 3.76 seconds, 0.6312 ms/image

Starting resource run 3/5
Run 3: 3.74 seconds, 0.6275 ms/image

Starting resource run 4/5
Run 4: 4.02 seconds, 0.6746 ms/image

Starting resource run 5/5
Run 5: 3.96 seconds, 0.6646 ms/image

Individual runs:


,elapsed_time_s,latency_ms_per_image,throughput_images_per_s,average_cpu_percent,peak_cpu_percent,baseline_ram_mb,average_ram_mb,peak_ram_mb,average_incremental_ram_mb,peak_incremental_ram_mb,...,average_gpu_memory_mb,peak_gpu_memory_mb,average_incremental_gpu_memory_mb,peak_incremental_gpu_memory_mb,average_gpu_utilization_percent,peak_gpu_utilization_percent,average_gpu_power_w,peak_gpu_power_w,gpu_energy_wh,run
0,3.642104,0.610989,1636.691409,3.155055,4.884375,6974.097656,7418.196806,7450.167969,444.099150,476.070312,...,6987.777344,6987.777344,0.0,0.0,68.500000,79,37.368000,42.431,0.037759,1
1,3.762877,0.631249,1584.160121,3.198162,5.193750,6976.410156,7417.557330,7452.304688,441.147174,475.894531,...,6987.777344,6987.777344,0.0,0.0,70.058824,94,33.921324,40.127,0.034368,2
2,3.740815,0.627548,1593.502892,3.226195,5.771875,6978.085938,7421.725873,7453.296875,443.639936,475.210938,...,6987.777344,6987.777344,0.0,0.0,68.794118,96,36.563324,40.579,0.037696,3
3,4.021120,0.674571,1482.422734,3.308420,5.771875,6978.457031,7423.655056,7455.042969,445.198025,476.585938,...,6987.777344,6987.777344,0.0,0.0,64.305556,98,36.220556,40.269,0.039728,4
4,3.961405,0.664554,1504.769179,3.091554,4.478125,6980.468750,7425.634291,7456.417969,445.165541,475.949219,...,6987.777344,6987.777344,0.0,0.0,62.135135,100,34.996838,40.119,0.038009,5


In [12]:
# ============================================================RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# ============================================================

metrics_to_report = [
    "elapsed_time_s",
    "latency_ms_per_image",
    "throughput_images_per_s",

    "average_cpu_percent",
    "peak_cpu_percent",

    "average_ram_mb",
    "peak_ram_mb",
    "average_incremental_ram_mb",
    "peak_incremental_ram_mb",

    "average_gpu_memory_mb",
    "peak_gpu_memory_mb",
    "average_incremental_gpu_memory_mb",
    "peak_incremental_gpu_memory_mb",

    "average_gpu_utilization_percent",
    "peak_gpu_utilization_percent",

    "average_gpu_power_w",
    "peak_gpu_power_w",
    "gpu_energy_wh"
]


summary_rows = []

number_of_runs = len(results_df)

for metric in metrics_to_report:
    values = results_df[metric].dropna()

    mean_value = values.mean()
    standard_deviation = values.std(ddof=1)

    if len(values) > 1:
        critical_t = t.ppf(
            0.975,
            df=len(values) - 1
        )

        confidence_half_width = (
            critical_t
            * standard_deviation
            / np.sqrt(len(values))
        )
    else:
        confidence_half_width = np.nan

    summary_rows.append({
        "Metric": metric,
        "Mean": mean_value,
        "Standard Deviation": standard_deviation,
        "95% CI Lower": (
            mean_value - confidence_half_width
        ),
        "95% CI Upper": (
            mean_value + confidence_half_width
        )
    })


resource_summary = pd.DataFrame(summary_rows)

print("\n" + "=" * 90)
print("EfficientNetV2_B0 RESOURCE CONSUMPTION — FIVE INFERENCE RUNS")
print("=" * 90)

display(resource_summary)


print("\nMain values for the manuscript")
print("-" * 90)

for metric in [
    "latency_ms_per_image",
    "peak_ram_mb",
    "peak_gpu_memory_mb",
    "average_gpu_utilization_percent",
    "average_gpu_power_w"
]:
    row = resource_summary[
        resource_summary["Metric"] == metric
    ].iloc[0]

    print(
        f"{metric}: "
        f"{row['Mean']:.3f} ± "
        f"{row['Standard Deviation']:.3f} "
        f"(95% CI: "
        f"{row['95% CI Lower']:.3f}–"
        f"{row['95% CI Upper']:.3f})"
    )


EfficientNetV2_B0 RESOURCE CONSUMPTION — FIVE INFERENCE RUNS


,Metric,Mean,Standard Deviation,95% CI Lower,95% CI Upper
0,elapsed_time_s,3.825664,0.159268,3.627908,4.023421
1,latency_ms_per_image,0.641782,0.026718,0.608607,0.674957
2,throughput_images_per_s,1560.309267,64.528918,1480.186033,1640.432501
3,average_cpu_percent,3.195877,0.080839,3.095502,3.296253
4,peak_cpu_percent,5.220000,0.564102,4.519575,5.920425
5,average_ram_mb,7421.353871,3.469028,7417.046504,7425.661238
6,peak_ram_mb,7453.446094,2.421427,7450.439495,7456.452692
7,average_incremental_ram_mb,443.849965,1.655241,441.794712,445.905218
8,peak_incremental_ram_mb,475.942187,0.491904,475.331408,476.552967
9,average_gpu_memory_mb,6987.777344,0.000000,6987.777344,6987.777344



Main values for the manuscript
------------------------------------------------------------------------------------------
latency_ms_per_image: 0.642 ± 0.027 (95% CI: 0.609–0.675)
peak_ram_mb: 7453.446 ± 2.421 (95% CI: 7450.439–7456.453)
peak_gpu_memory_mb: 6987.777 ± 0.000 (95% CI: 6987.777–6987.777)
average_gpu_utilization_percent: 66.759 ± 3.371 (95% CI: 62.573–70.945)
average_gpu_power_w: 35.814 ± 1.359 (95% CI: 34.126–37.502)


#genralization

In [14]:
#wild deepfake on celeb
test_results = model.evaluate(
    test_images,
    test_labels,
    batch_size=16,
    verbose=1,
    return_dict=True
)

print("\nTest results of Celeb-DF(V2) on wild deepfake dataset (EfficientNetV2_B0):")
for metric_name, metric_value in test_results.items():
    print(f"{metric_name}: {metric_value:.4f}")


1125/1125 [==============================] - 17s 15ms/step - loss: 0.6276 - accuracy: 0.7501 - precision: 0.7500 - recall: 1.0000 - roc_auc: 0.4693 - pr_auc: 0.7223

Test results of Celeb-DF(V2) on wild deepfake dataset (EfficientNetV2_B0):
loss: 0.6276
accuracy: 0.7501
precision: 0.7500
recall: 1.0000
roc_auc: 0.4693
pr_auc: 0.7223


In [16]:
#DFC on celeb
test_results = model.evaluate(
    test_hog,
    test_labels,
    batch_size=16,
    verbose=1,
    return_dict=True
)

print("\nTest results of Celeb-DF(V2) on DFC dataset (EfficientNetV2_B0):")
for metric_name, metric_value in test_results.items():
    print(f"{metric_name}: {metric_value:.4f}")



188/188 [==============================] - 5s 16ms/step - loss: 0.9339 - accuracy: 0.4993 - precision: 0.4997 - recall: 0.9960 - roc_auc: 0.4674 - pr_auc: 0.4661

Test results of Celeb-DF(V2) on DFC dataset (EfficientNetV2_B0):
loss: 0.9339
accuracy: 0.4993
precision: 0.4997
recall: 0.9960
roc_auc: 0.4674
pr_auc: 0.4661


In [18]:
#FF++ on celeb
test_results = model.evaluate(
    test_ff,
    test_ff_labels,
    batch_size=16,
    verbose=1,
    return_dict=True
)
print("\nTest results of Celeb-DF(V2) on FF++ dataset (EfficientNetV2_B0):")
for metric_name, metric_value in test_results.items():
    print(f"{metric_name}: {metric_value:.4f}")


155/155 [==============================] - 2s 12ms/step - loss: 1.0767 - accuracy: 0.4178 - precision: 0.4168 - recall: 0.9961 - roc_auc: 0.5276 - pr_auc: 0.4538

Test results of Celeb-DF(V2) on FF++ dataset (EfficientNetV2_B0):
loss: 1.0767
accuracy: 0.4178
precision: 0.4168
recall: 0.9961
roc_auc: 0.5276
pr_auc: 0.4538


# DFC

In [4]:
import h5py
import numpy as np
# Open the HDF5 file in read mode
with h5py.File('D://thesis//dataset//deepfake dataset//resized_images.h5', 'r') as h5f:
    # Access each dataset
    celeb = np.array(h5f['celeb'])
    ffhq = np.array(h5f['ffhq'])
    gdwct = np.array(h5f['gdwct'])
    attgan = np.array(h5f['attgan'])
    stargan = np.array(h5f['stargan'])
    stylegan2 = np.array(h5f['stylegan2'])
    stylegan = np.array(h5f['stylegan'])

# Now, 'celeb', 'ffhq', etc., are NumPy arrays containing your datasets
print(f"celeb shape: {celeb.shape}, dtype: {celeb.dtype}")
print(f"ffhq shape: {ffhq.shape}, dtype: {ffhq.dtype}")
print(f"ffhq shape: {gdwct.shape}, dtype: {gdwct.dtype}")
print(f"ffhq shape: {attgan.shape}, dtype: {attgan.dtype}")
print(f"ffhq shape: {stargan.shape}, dtype: {stargan.dtype}")
print(f"ffhq shape: {stylegan2.shape}, dtype: {stylegan2.dtype}")
print(f"ffhq shape: {stylegan.shape}, dtype: {stylegan.dtype}")
# Repeat for other datasets as needed
import cv2
# Function to resize images from (224, 224) to (160, 160)
def resize_images(image_array, target_size=(160, 160)):
    resized_images = np.array([cv2.resize(img, target_size) for img in image_array])
    return resized_images

celeb = resize_images(celeb, target_size=(160, 160))
ffhq = resize_images(ffhq, target_size=(160, 160))
gdwct = resize_images(gdwct, target_size=(160, 160))
attgan = resize_images(attgan, target_size=(160, 160))
stargan = resize_images(stargan, target_size=(160, 160))
stylegan = resize_images(stylegan, target_size=(160, 160))
stylegan2 = resize_images(stylegan2, target_size=(160, 160))
import random
# Randomly select 2500 distinct images
random_indices = random.sample(range(len(celeb)), 2500)  # Get 2500 random indices
celeb = celeb[random_indices]  # Select the random subse

import random
# Randomly select 2500 distinct images
random_indices = random.sample(range(len(ffhq)), 2500)  # Get 2500 random indices
ffhq = ffhq[random_indices]  # Select the random subse
print(f"celeb shape: {celeb.shape}, dtype: {celeb.dtype}")
print(f"ffhq shape: {ffhq.shape}, dtype: {ffhq.dtype}")
print(f"gdwct shape: {gdwct.shape}, dtype: {gdwct.dtype}")
print(f"attagan shape: {attgan.shape}, dtype: {attgan.dtype}")
print(f"stargan shape: {stargan.shape}, dtype: {stargan.dtype}")
print(f"stylegan2 shape: {stylegan2.shape}, dtype: {stylegan2.dtype}")
print(f"stylegan shape: {stylegan.shape}, dtype: {stylegan.dtype}")
import random
import numpy as np

def split_data(data, train_ratio=0.7):
    """
    Splits data into training and testing sets based on the specified ratio.

    Parameters:
        data (list or np.array): The dataset to split.
        train_ratio (float): The ratio of the data to include in the training set.

    Returns:
        tuple: Two datasets - train and test.
    """
    # Shuffle the data
    random.shuffle(data)

    # Calculate the split index
    split_index = int(len(data) * train_ratio)

    # Split the data
    train_data = data[:split_index]
    test_data = data[split_index:]

    return train_data, test_data

# Split `celeb` into 70% train and 30% test
celeb_train_hog, celeb_test_hog = split_data(celeb, train_ratio=0.7)

# Split `ffhq` into 70% train and 30% test
ffhq_train_hog, ffhq_test_hog = split_data(ffhq, train_ratio=0.7)

# Split `attgan` into 70% train and 30% test
attgan_train_hog, attgan_test_hog = split_data(attgan, train_ratio=0.7)

# Split `stargan` into 70% train and 30% test
stargan_train_hog, stargan_test_hog = split_data(stargan, train_ratio=0.7)

# Split `gdwct` into 70% train and 30% test
gdwct_train_hog, gdwct_test_hog = split_data(gdwct, train_ratio=0.7)

# Split `stylegan2` into 70% train and 30% test_hog
stylegan2_train_hog, stylegan2_test_hog = split_data(stylegan2, train_ratio=0.7)

# Split `stylegan` into 70% train and 30% test_hog
stylegan_train_hog, stylegan_test_hog = split_data(stylegan, train_ratio=0.7)

# Convert to NumPy arrays if needed
celeb_train_hog, celeb_test_hog = np.array(celeb_train_hog), np.array(celeb_test_hog)
ffhq_train_hog, ffhq_test_hog = np.array(ffhq_train_hog), np.array(ffhq_test_hog)
attgan_train_hog, attgan_test_hog = np.array(attgan_train_hog), np.array(attgan_test_hog)
stargan_train_hog, stargan_test_hog = np.array(stargan_train_hog), np.array(stargan_test_hog)
gdwct_train_hog, gdwct_test_hog = np.array(gdwct_train_hog), np.array(gdwct_test_hog)
stylegan2_train_hog, stylegan2_test_hog = np.array(stylegan2_train_hog), np.array(stylegan2_test_hog)
stylegan_train_hog, stylegan_test_hog = np.array(stylegan_train_hog), np.array(stylegan_test_hog)

# Print results for verification
print(f"celeb_train: {len(celeb_train_hog)} images, celeb_test: {len(celeb_test_hog)} images")
print(f"ffhq_train: {len(ffhq_train_hog)} images, ffhq_test: {len(ffhq_test_hog)} images")
print(f"attgan_train: {len(attgan_train_hog)} images, attgan_test: {len(attgan_test_hog)} images")
print(f"stargan_train: {len(stargan_train_hog)} images, stargan_test: {len(stargan_test_hog)} images")
print(f"gdwct_train: {len(gdwct_train_hog)} images, gdwct_test: {len(gdwct_test_hog)} images")
print(f"stylegan2_train: {len(stylegan2_train_hog)} images, stylegan2_test: {len(stylegan2_test_hog)} images")
print(f"stylegan_train: {len(stylegan_train_hog)} images, stylegan_test: {len(stylegan_test_hog)} images")

########################################################################################################################################
#######################################divide into 60,10 train and val
#########################################################################################################################################
def extract_validation(train_data):
    """
    Extract every 10th sample from the training data and store it in a validation set.

    Parameters:
        train_data (list or np.array): The training dataset.

    Returns:
        tuple: Updated training dataset and validation dataset.
    """
    # Select every 10th sample for the validation set
    validation_data = train_data[::10]

    # Remove the selected samples from the training dataset
    updated_train_data = [train_data[i] for i in range(len(train_data)) if i % 10 != 0]

    return np.array(updated_train_data), np.array(validation_data)


# Perform the operation for each dataset
celeb_train_hog, celeb_val_hog = extract_validation(celeb_train_hog)
ffhq_train_hog, ffhq_val_hog = extract_validation(ffhq_train_hog)
attgan_train_hog, attgan_val_hog = extract_validation(attgan_train_hog)
stargan_train_hog, stargan_val_hog = extract_validation(stargan_train_hog)
gdwct_train_hog, gdwct_val_hog = extract_validation(gdwct_train_hog)
stylegan2_train_hog, stylegan2_val_hog = extract_validation(stylegan2_train_hog)
stylegan_train_hog, stylegan_val_hog = extract_validation(stylegan_train_hog)

# Print results for verification
print(f"celeb_train: {len(celeb_train_hog)} images, celeb_val: {len(celeb_val_hog)} images")
print(f"ffhq_train: {len(ffhq_train_hog)} images, ffhq_val: {len(ffhq_val_hog)} images")
print(f"attgan_train: {len(attgan_train_hog)} images, attgan_val: {len(attgan_val_hog)} images")
print(f"stargan_train: {len(stargan_train_hog)} images, stargan_val: {len(stargan_val_hog)} images")
print(f"gdwct_train: {len(gdwct_train_hog)} images, gdwct_val: {len(gdwct_val_hog)} images")
print(f"stylegan2_train: {len(stylegan2_train_hog)} images, stylegan2_val: {len(stylegan2_val_hog)} images")
print(f"stylegan_train: {len(stylegan_train_hog)} images, stylegan_val: {len(stylegan_val_hog)} images")
############################################################################################################################################################
#################################################concatenate the labels 0,1 real and fake
#############################################################################################################################################################


celeb_train_labels = np.zeros(len(celeb_train_hog), dtype=int)
ffhq_train_labels = np.zeros(len(ffhq_train_hog), dtype=int)
atta_train_labels = np.ones(len(attgan_train_hog), dtype=int)
star_train_labels = np.ones(len(stargan_train_hog), dtype=int)
gdwct_train_labels = np.ones(len(gdwct_train_hog), dtype=int)
stylegan2_train_labels = np.ones(len(stylegan2_train_hog), dtype=int)
stylegan_train_labels = np.ones(len(stylegan_train_hog), dtype=int)

# Concatenate all training datasets into a single `train` variable
train_hog = np.concatenate([celeb_train_hog, ffhq_train_hog, attgan_train_hog, stargan_train_hog, gdwct_train_hog, stylegan2_train_hog, stylegan_train_hog], axis=0)
train_labels=np.concatenate([celeb_train_labels, ffhq_train_labels, atta_train_labels, star_train_labels, gdwct_train_labels, stylegan2_train_labels,
                              stylegan_train_labels], axis=0)




celeb_test_labels = np.zeros(len(celeb_test_hog), dtype=int)
ffhq_test_labels = np.zeros(len(ffhq_test_hog), dtype=int)
atta_test_labels = np.ones(len(attgan_test_hog), dtype=int)
star_test_labels = np.ones(len(stargan_test_hog), dtype=int)
gdwct_test_labels = np.ones(len(gdwct_test_hog), dtype=int)
stylegan2_test_labels = np.ones(len(stylegan2_test_hog), dtype=int)
stylegan_test_labels = np.ones(len(stylegan_test_hog), dtype=int)

# Concatenate all testing datasets into a single `test` variable
test_hog = np.concatenate([celeb_test_hog, ffhq_test_hog, attgan_test_hog, stargan_test_hog, gdwct_test_hog, stylegan2_test_hog, stylegan_test_hog], axis=0)
test_labels = np.concatenate([celeb_test_labels, ffhq_test_labels, atta_test_labels, star_test_labels, gdwct_test_labels, stylegan2_test_labels,
                        stylegan_test_labels], axis=0)




celeb_val_labels = np.zeros(len(celeb_val_hog), dtype=int)
ffhq_val_labels = np.zeros(len(ffhq_val_hog), dtype=int)
atta_val_labels = np.ones(len(attgan_val_hog), dtype=int)
star_val_labels = np.ones(len(stargan_val_hog), dtype=int)
gdwct_val_labels = np.ones(len(gdwct_val_hog), dtype=int)
stylegan2_val_labels = np.ones(len(stylegan2_val_hog), dtype=int)
stylegan_val_labels = np.ones(len(stylegan_val_hog), dtype=int)

# Concatenate all validation datasets into a single `val` variable
val_hog = np.concatenate([celeb_val_hog, ffhq_val_hog, attgan_val_hog, stargan_val_hog, gdwct_val_hog, stylegan2_val_hog, stylegan_val_hog], axis=0)
val_labels = np.concatenate([celeb_val_labels, ffhq_val_labels, atta_val_labels, star_val_labels, gdwct_val_labels, stylegan2_val_labels,
                       stylegan_val_labels], axis=0)

# Print the results for verification
print(f"Total train: {len(train_hog)} images")
print(f"Total test: {len(test_hog)} images")
print(f"Total val: {len(val_hog)} images")


# Print results for verification
print(f"Train Labels: {len(train_labels)} ")
print(f"Test Labels: {len(test_labels)} ")
print(f"Val Labels: {len(val_labels)} ")



celeb shape: (5000, 224, 224, 3), dtype: uint8
ffhq shape: (5000, 224, 224, 3), dtype: uint8
ffhq shape: (1000, 224, 224, 3), dtype: uint8
ffhq shape: (1000, 224, 224, 3), dtype: uint8
ffhq shape: (1000, 224, 224, 3), dtype: uint8
ffhq shape: (1000, 224, 224, 3), dtype: uint8
ffhq shape: (1000, 224, 224, 3), dtype: uint8
celeb shape: (2500, 160, 160, 3), dtype: uint8
ffhq shape: (2500, 160, 160, 3), dtype: uint8
gdwct shape: (1000, 160, 160, 3), dtype: uint8
attagan shape: (1000, 160, 160, 3), dtype: uint8
stargan shape: (1000, 160, 160, 3), dtype: uint8
stylegan2 shape: (1000, 160, 160, 3), dtype: uint8
stylegan shape: (1000, 160, 160, 3), dtype: uint8
celeb_train: 1750 images, celeb_test: 750 images
ffhq_train: 1750 images, ffhq_test: 750 images
attgan_train: 700 images, attgan_test: 300 images
stargan_train: 700 images, stargan_test: 300 images
gdwct_train: 700 images, gdwct_test: 300 images
stylegan2_train: 700 images, stylegan2_test: 300 images
stylegan_train: 700 images, stylegan

In [5]:
# Train model efficent v2-b0
history = model.fit(
    train_hog, train_labels,
    validation_data=(val_hog, val_labels),
    epochs=10,
    batch_size=16,
    verbose=1
)

Epoch 1/10
394/394 [==============================] - 23s 40ms/step - loss: 0.5657 - accuracy: 0.7021 - precision: 0.6972 - recall: 0.7143 - roc_auc: 0.7768 - pr_auc: 0.7686 - val_loss: 0.4173 - val_accuracy: 0.8257 - val_precision: 0.8032 - val_recall: 0.8629 - val_roc_auc: 0.8950 - val_pr_auc: 0.8894
Epoch 2/10
394/394 [==============================] - 15s 39ms/step - loss: 0.3598 - accuracy: 0.8425 - precision: 0.8391 - recall: 0.8476 - roc_auc: 0.9202 - pr_auc: 0.9173 - val_loss: 0.3026 - val_accuracy: 0.8743 - val_precision: 0.8876 - val_recall: 0.8571 - val_roc_auc: 0.9505 - val_pr_auc: 0.9538
Epoch 3/10
394/394 [==============================] - 16s 40ms/step - loss: 0.2083 - accuracy: 0.9189 - precision: 0.9172 - recall: 0.9210 - roc_auc: 0.9736 - pr_auc: 0.9742 - val_loss: 0.2233 - val_accuracy: 0.9271 - val_precision: 0.8966 - val_recall: 0.9657 - val_roc_auc: 0.9761 - val_pr_auc: 0.9675
Epoch 4/10
394/394 [==============================] - 16s 40ms/step - loss: 0.1197 - acc

In [ ]:
print("=== DATA LOADING ===")
start = monitor.get_stats()



=== DATA LOADING ===


In [ ]:
# ============================================================
# COMPLETE TEST EVALUATION
# Labels: 0 = Real, 1 = Fake
# ============================================================

import numpy as np

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    roc_curve,
    auc,
    confusion_matrix,
    classification_report,
    matthews_corrcoef
)

BATCH_SIZE = 16
DECISION_THRESHOLD = 0.50


# ------------------------------------------------------------
# 1. Test loss
# ------------------------------------------------------------

evaluation = model.evaluate(
    test_hog,
    test_labels,
    batch_size=BATCH_SIZE,
    verbose=1,
    return_dict=True
)

test_loss = evaluation["loss"]


# ------------------------------------------------------------
# 2. Prediction probabilities and binary predictions
# ------------------------------------------------------------

test_probabilities = model.predict(
    test_hog,
    batch_size=BATCH_SIZE,
    verbose=1
).reshape(-1)

y_true = np.asarray(test_labels).reshape(-1).astype(np.uint8)

y_pred = (
    test_probabilities >= DECISION_THRESHOLD
).astype(np.uint8)


# ------------------------------------------------------------
# 3. Confusion matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1]
)

tn, fp, fn, tp = cm.ravel()


# ------------------------------------------------------------
# 4. Threshold-dependent metrics
# ------------------------------------------------------------

accuracy = accuracy_score(y_true, y_pred)

balanced_accuracy = balanced_accuracy_score(
    y_true,
    y_pred
)

precision = precision_score(
    y_true,
    y_pred,
    pos_label=1,
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    pos_label=1,
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    pos_label=1,
    zero_division=0
)

specificity = (
    tn / (tn + fp)
    if (tn + fp) > 0
    else 0.0
)

false_positive_rate_at_05 = (
    fp / (fp + tn)
    if (fp + tn) > 0
    else 0.0
)

false_negative_rate_at_05 = (
    fn / (fn + tp)
    if (fn + tp) > 0
    else 0.0
)

mcc = matthews_corrcoef(
    y_true,
    y_pred
)


# ------------------------------------------------------------
# 5. ROC-AUC
# ------------------------------------------------------------

roc_auc = roc_auc_score(
    y_true,
    test_probabilities
)

fpr, tpr, roc_thresholds = roc_curve(
    y_true,
    test_probabilities,
    pos_label=1
)


# ------------------------------------------------------------
# 6. PR-AUC and Average Precision
# ------------------------------------------------------------

pr_precision, pr_recall, _ = precision_recall_curve(
    y_true,
    test_probabilities,
    pos_label=1
)

# Reverse because recall is normally returned in descending order
pr_auc = auc(
    pr_recall[::-1],
    pr_precision[::-1]
)

average_precision = average_precision_score(
    y_true,
    test_probabilities
)


# ------------------------------------------------------------
# 7. Equal Error Rate
# ------------------------------------------------------------

fnr_curve = 1.0 - tpr

# Remove non-finite thresholds such as infinity
valid_indices = np.where(
    np.isfinite(roc_thresholds)
)[0]

eer_index = valid_indices[
    np.argmin(
        np.abs(
            fpr[valid_indices]
            - fnr_curve[valid_indices]
        )
    )
]

eer = (
    fpr[eer_index]
    + fnr_curve[eer_index]
) / 2.0

eer_threshold = roc_thresholds[eer_index]


# ------------------------------------------------------------
# 8. Display all results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("EfficientNetB0 — COMPLETE TEST-SET EVALUATION")
print("=" * 70)

print(f"Number of test samples:     {len(y_true)}")
print(f"Decision threshold:         {DECISION_THRESHOLD:.4f}")
print(f"Test loss:                  {test_loss:.6f}")

print("\nMain evaluation metrics")
print("-" * 70)
print(f"Accuracy:                   {accuracy:.6f} ({accuracy*100:.2f}%)")
print(f"Precision:                  {precision:.6f} ({precision*100:.2f}%)")
print(f"Recall/Sensitivity:         {recall:.6f} ({recall*100:.2f}%)")
print(f"F1-score:                   {f1:.6f} ({f1*100:.2f}%)")
print(f"ROC-AUC:                    {roc_auc:.6f}")
print(f"PR-AUC:                     {pr_auc:.6f}")
print(f"Average Precision:          {average_precision:.6f}")
print(f"EER:                        {eer:.6f} ({eer*100:.2f}%)")
print(f"EER threshold:              {eer_threshold:.6f}")

print("\nAdditional evaluation metrics")
print("-" * 70)
print(f"Balanced accuracy:          {balanced_accuracy:.6f}")
print(f"Specificity:                {specificity:.6f}")
print(f"Matthews correlation:       {mcc:.6f}")
print(f"False-positive rate @ 0.5:  {false_positive_rate_at_05:.6f}")
print(f"False-negative rate @ 0.5:  {false_negative_rate_at_05:.6f}")

print("\nConfusion matrix")
print("-" * 70)
print("Rows = actual classes; columns = predicted classes")
print("Class order: [Real, Fake]")
print(cm)

print("\nConfusion-matrix values")
print("-" * 70)
print(f"True Negative  — Real predicted as Real: {tn}")
print(f"False Positive — Real predicted as Fake: {fp}")
print(f"False Negative — Fake predicted as Real: {fn}")
print(f"True Positive  — Fake predicted as Fake: {tp}")

print("\nClassification report")
print("-" * 70)

print(
    classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=["Real", "Fake"],
        digits=6,
        zero_division=0
    )
)


EfficientNetV2_B0 — COMPLETE TEST-SET EVALUATION
Number of test samples:     3000
Decision threshold:         0.5000
Test loss:                  0.157173

Main evaluation metrics
----------------------------------------------------------------------
Accuracy:                   0.951000 (95.10%)
Precision:                  0.980128 (98.01%)
Recall/Sensitivity:         0.920667 (92.07%)
F1-score:                   0.949467 (94.95%)
ROC-AUC:                    0.992141
PR-AUC:                     0.993284
Average Precision:          0.993287
EER:                        0.039333 (3.93%)
EER threshold:              0.085295

Additional evaluation metrics
----------------------------------------------------------------------
Balanced accuracy:          0.951000
Specificity:                0.981333
Matthews correlation:       0.903664
False-positive rate @ 0.5:  0.018667
False-negative rate @ 0.5:  0.079333

Confusion matrix
-------------------------------------------------------------------

In [ ]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"RAM Used: {end['ram_mb'] - start['ram_mb']:.1f} MB")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts

save the model

In [ ]:
MODEL_PATH = (
    r"D:\thesis\results"
    r"\EfficientNetB0_dfc_10epochs.h5"
)

model.save(
    MODEL_PATH,
    include_optimizer=True
)

print("Model saved successfully:")
print(MODEL_PATH)
# After training the model
model.save('EfficientNetB0_160_dfc.h5')  # Saves the entire model to a file

Model saved successfully:
D:\thesis\results\Inception_V3_dfc_10epochs.h5


#load the model

In [ ]:
import h5py
import tensorflow as tf

MODEL_PATH = ( r"D:\thesis\results" r"\EfficientNetB0_dfc_10epochs.h5")


model = tf.keras.models.load_model(
    MODEL_PATH,
    compile=False
)



In [ ]:
import h5py
import tensorflow as tf

MODEL_PATH = ( r"EfficientNetB0_160_dfc.h5")


In [8]:
# ============================================================
# REPEATED INFERENCE RESOURCE PROFILING
# ============================================================

import os
import gc
import time
import threading
import numpy as np
import pandas as pd
import psutil

from scipy.stats import t

from pynvml import (
    nvmlInit,
    nvmlShutdown,
    nvmlDeviceGetHandleByIndex,
    nvmlDeviceGetMemoryInfo,
    nvmlDeviceGetUtilizationRates,
    nvmlDeviceGetPowerUsage,
    NVMLError
)


BATCH_SIZE = 16
NUMBER_OF_RUNS = 5
SAMPLING_INTERVAL = 0.1     # Sample every 100 milliseconds
COOLDOWN_SECONDS = 5
GPU_INDEX = 0


class ResourceMonitor:
    """
    Continuously measures resource consumption during one inference run.

    RAM:
        Current Python process resident memory (RSS).

    GPU memory:
        Total device memory currently used, with the pre-inference
        baseline subtracted for incremental measurements.

    CPU:
        Current Python-process CPU utilization normalized by the
        number of logical CPU cores.

    Power:
        NVIDIA GPU board power reported by NVML.
    """

    def __init__(
        self,
        interval=0.1,
        gpu_index=0
    ):
        self.interval = interval
        self.process = psutil.Process(os.getpid())

        self.logical_cpu_count = (
            psutil.cpu_count(logical=True) or 1
        )

        self.stop_event = threading.Event()
        self.samples = []
        self.thread = None

        nvmlInit()
        self.gpu_handle = nvmlDeviceGetHandleByIndex(
            gpu_index
        )

    def _read_gpu_power(self):
        try:
            return (
                nvmlDeviceGetPowerUsage(
                    self.gpu_handle
                ) / 1000.0
            )
        except NVMLError:
            return np.nan

    def _collect_sample(self):
        timestamp = time.perf_counter()

        # Process RAM: resident set size
        ram_mb = (
            self.process.memory_info().rss
            / (1024 ** 2)
        )

        # Process CPU percentage can exceed 100% on multicore systems.
        # Normalize it to an approximate 0–100% scale.
        process_cpu_raw = self.process.cpu_percent(
            interval=None
        )

        process_cpu_normalized = (
            process_cpu_raw
            / self.logical_cpu_count
        )

        gpu_memory = nvmlDeviceGetMemoryInfo(
            self.gpu_handle
        )

        gpu_memory_used_mb = (
            gpu_memory.used
            / (1024 ** 2)
        )

        gpu_utilization = nvmlDeviceGetUtilizationRates(
            self.gpu_handle
        ).gpu

        gpu_power_w = self._read_gpu_power()

        self.samples.append({
            "timestamp": timestamp,
            "ram_mb": ram_mb,
            "cpu_percent": process_cpu_normalized,
            "gpu_memory_mb": gpu_memory_used_mb,
            "gpu_utilization_percent": gpu_utilization,
            "gpu_power_w": gpu_power_w
        })

    def _sampling_loop(self):
        while not self.stop_event.is_set():
            try:
                self._collect_sample()
            except Exception as error:
                print("Monitoring warning:", error)

            self.stop_event.wait(self.interval)

    def start(self):
        # Initialize CPU counters
        self.process.cpu_percent(interval=None)

        # First sample is the pre-inference baseline
        self._collect_sample()

        self.thread = threading.Thread(
            target=self._sampling_loop,
            daemon=True
        )

        self.thread.start()

    def stop(
        self,
        elapsed_seconds,
        number_of_images
    ):
        self.stop_event.set()

        if self.thread is not None:
            self.thread.join()

        # Capture one final sample
        try:
            self._collect_sample()
        except Exception:
            pass

        nvmlShutdown()

        data = pd.DataFrame(self.samples)

        baseline_ram = data["ram_mb"].iloc[0]
        baseline_gpu_memory = data[
            "gpu_memory_mb"
        ].iloc[0]

        peak_ram = data["ram_mb"].max()
        average_ram = data["ram_mb"].mean()

        peak_gpu_memory = data[
            "gpu_memory_mb"
        ].max()

        average_gpu_memory = data[
            "gpu_memory_mb"
        ].mean()

        peak_incremental_ram = max(
            0.0,
            peak_ram - baseline_ram
        )

        average_incremental_ram = max(
            0.0,
            average_ram - baseline_ram
        )

        peak_incremental_gpu_memory = max(
            0.0,
            peak_gpu_memory - baseline_gpu_memory
        )

        average_incremental_gpu_memory = max(
            0.0,
            average_gpu_memory - baseline_gpu_memory
        )

        # Estimate GPU energy using power integration
        valid_power = data.dropna(
            subset=["gpu_power_w"]
        )

        if len(valid_power) >= 2:
            relative_times = (
                valid_power["timestamp"].to_numpy()
                - valid_power["timestamp"].iloc[0]
            )

            energy_joules = np.trapz(
                valid_power["gpu_power_w"].to_numpy(),
                relative_times
            )

            energy_wh = energy_joules / 3600.0
        else:
            energy_wh = np.nan

        return {
            "elapsed_time_s": elapsed_seconds,

            "latency_ms_per_image": (
                elapsed_seconds
                / number_of_images
                * 1000.0
            ),

            "throughput_images_per_s": (
                number_of_images
                / elapsed_seconds
            ),

            "average_cpu_percent": (
                data["cpu_percent"].mean()
            ),

            "peak_cpu_percent": (
                data["cpu_percent"].max()
            ),

            "baseline_ram_mb": baseline_ram,
            "average_ram_mb": average_ram,
            "peak_ram_mb": peak_ram,

            "average_incremental_ram_mb": (
                average_incremental_ram
            ),

            "peak_incremental_ram_mb": (
                peak_incremental_ram
            ),

            "baseline_gpu_memory_mb": (
                baseline_gpu_memory
            ),

            "average_gpu_memory_mb": (
                average_gpu_memory
            ),

            "peak_gpu_memory_mb": (
                peak_gpu_memory
            ),

            "average_incremental_gpu_memory_mb": (
                average_incremental_gpu_memory
            ),

            "peak_incremental_gpu_memory_mb": (
                peak_incremental_gpu_memory
            ),

            "average_gpu_utilization_percent": (
                data["gpu_utilization_percent"].mean()
            ),

            "peak_gpu_utilization_percent": (
                data["gpu_utilization_percent"].max()
            ),

            "average_gpu_power_w": (
                data["gpu_power_w"].mean()
            ),

            "peak_gpu_power_w": (
                data["gpu_power_w"].max()
            ),

            "gpu_energy_wh": energy_wh
        }


# ------------------------------------------------------------
# GPU/model warm-up
# ------------------------------------------------------------

warmup_count = min(
    len(test_hog),
    BATCH_SIZE * 3
)

print("Performing warm-up inference...")

_ = model.predict(
    test_hog[:warmup_count],
    batch_size=BATCH_SIZE,
    verbose=0
)

print("Warm-up completed.")


# ------------------------------------------------------------
# Five repeated inference runs
# ------------------------------------------------------------

run_results = []

for run_number in range(
    1,
    NUMBER_OF_RUNS + 1
):
    print(
        f"\nStarting resource run "
        f"{run_number}/{NUMBER_OF_RUNS}"
    )

    gc.collect()
    time.sleep(COOLDOWN_SECONDS)

    monitor = ResourceMonitor(
        interval=SAMPLING_INTERVAL,
        gpu_index=GPU_INDEX
    )

    monitor.start()

    start_time = time.perf_counter()

    predictions = model.predict(
        test_hog,
        batch_size=BATCH_SIZE,
        verbose=0
    )

    elapsed_time = (
        time.perf_counter()
        - start_time
    )

    # Access the result to ensure it is materialized
    _ = float(predictions[-1].reshape(-1)[0])

    run_summary = monitor.stop(
        elapsed_seconds=elapsed_time,
        number_of_images=len(test_hog)
    )

    run_summary["run"] = run_number
    run_results.append(run_summary)

    print(
        f"Run {run_number}: "
        f"{elapsed_time:.2f} seconds, "
        f"{run_summary['latency_ms_per_image']:.4f} ms/image"
    )

    del predictions


results_df = pd.DataFrame(run_results)

print("\nIndividual runs:")
display(results_df)

C:\Users\aneek\AppData\Local\Temp\ipykernel_4048\3709879675.py:15: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  from pynvml import (


Performing warm-up inference...
Warm-up completed.

Starting resource run 1/5
Run 1: 2.03 seconds, 0.6753 ms/image

Starting resource run 2/5
Run 2: 2.03 seconds, 0.6759 ms/image

Starting resource run 3/5
Run 3: 2.07 seconds, 0.6904 ms/image

Starting resource run 4/5
Run 4: 1.99 seconds, 0.6630 ms/image

Starting resource run 5/5
Run 5: 2.05 seconds, 0.6819 ms/image

Individual runs:


,elapsed_time_s,latency_ms_per_image,throughput_images_per_s,average_cpu_percent,peak_cpu_percent,baseline_ram_mb,average_ram_mb,peak_ram_mb,average_incremental_ram_mb,peak_incremental_ram_mb,...,average_gpu_memory_mb,peak_gpu_memory_mb,average_incremental_gpu_memory_mb,peak_incremental_gpu_memory_mb,average_gpu_utilization_percent,peak_gpu_utilization_percent,average_gpu_power_w,peak_gpu_power_w,gpu_energy_wh,run
0,2.025776,0.675259,1480.914054,3.332656,9.765625,6256.035156,6483.337305,6517.050781,227.302148,261.015625,...,7300.917969,7300.917969,0.0,0.0,59.700000,86,33.799700,41.050,0.019234,1
1,2.027626,0.675875,1479.563091,3.180000,4.928125,6259.144531,6486.539844,6519.046875,227.395312,259.902344,...,7300.917969,7300.917969,0.0,0.0,61.750000,95,33.796050,40.103,0.019087,2
2,2.071121,0.690374,1448.490798,2.880000,4.031250,6261.617188,6487.574219,6520.449219,225.957031,258.832031,...,7300.917969,7300.917969,0.0,0.0,61.000000,95,23.766050,40.050,0.013365,3
3,1.989047,0.663016,1508.259758,2.968421,4.478125,6262.417969,6488.010074,6520.570312,225.592105,258.152344,...,7300.917969,7300.917969,0.0,0.0,61.315789,86,33.484474,40.966,0.018692,4
4,2.045635,0.681878,1466.537357,3.054219,4.478125,6262.726562,6484.519727,6520.878906,221.793164,258.152344,...,7300.917969,7300.917969,0.0,0.0,63.600000,95,34.145750,41.241,0.019407,5


In [9]:
# ============================================================RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# ============================================================

metrics_to_report = [
    "elapsed_time_s",
    "latency_ms_per_image",
    "throughput_images_per_s",

    "average_cpu_percent",
    "peak_cpu_percent",

    "average_ram_mb",
    "peak_ram_mb",
    "average_incremental_ram_mb",
    "peak_incremental_ram_mb",

    "average_gpu_memory_mb",
    "peak_gpu_memory_mb",
    "average_incremental_gpu_memory_mb",
    "peak_incremental_gpu_memory_mb",

    "average_gpu_utilization_percent",
    "peak_gpu_utilization_percent",

    "average_gpu_power_w",
    "peak_gpu_power_w",
    "gpu_energy_wh"
]


summary_rows = []

number_of_runs = len(results_df)

for metric in metrics_to_report:
    values = results_df[metric].dropna()

    mean_value = values.mean()
    standard_deviation = values.std(ddof=1)

    if len(values) > 1:
        critical_t = t.ppf(
            0.975,
            df=len(values) - 1
        )

        confidence_half_width = (
            critical_t
            * standard_deviation
            / np.sqrt(len(values))
        )
    else:
        confidence_half_width = np.nan

    summary_rows.append({
        "Metric": metric,
        "Mean": mean_value,
        "Standard Deviation": standard_deviation,
        "95% CI Lower": (
            mean_value - confidence_half_width
        ),
        "95% CI Upper": (
            mean_value + confidence_half_width
        )
    })


resource_summary = pd.DataFrame(summary_rows)

print("\n" + "=" * 90)
print("EfficientNetV2-B0 RESOURCE CONSUMPTION — FIVE INFERENCE RUNS")
print("=" * 90)

display(resource_summary)


print("\nMain values for the manuscript")
print("-" * 90)

for metric in [
    "latency_ms_per_image",
    "peak_ram_mb",
    "peak_gpu_memory_mb",
    "average_gpu_utilization_percent",
    "average_gpu_power_w"
]:
    row = resource_summary[
        resource_summary["Metric"] == metric
    ].iloc[0]

    print(
        f"{metric}: "
        f"{row['Mean']:.3f} ± "
        f"{row['Standard Deviation']:.3f} "
        f"(95% CI: "
        f"{row['95% CI Lower']:.3f}–"
        f"{row['95% CI Upper']:.3f})"
    )


EfficientNetV2-B0 RESOURCE CONSUMPTION — FIVE INFERENCE RUNS


,Metric,Mean,Standard Deviation,95% CI Lower,95% CI Upper
0,elapsed_time_s,2.031841,0.030079,1.994493,2.069189
1,latency_ms_per_image,0.677280,0.010026,0.664831,0.689730
2,throughput_images_per_s,1476.753012,21.914597,1449.542446,1503.963578
3,average_cpu_percent,3.083059,0.178117,2.861898,3.304221
4,peak_cpu_percent,5.536250,2.385462,2.574308,8.498192
5,average_ram_mb,6485.996234,2.005684,6483.505848,6488.486620
6,peak_ram_mb,6519.599219,1.589240,6517.625916,6521.572521
7,average_incremental_ram_mb,225.607952,2.277042,222.780632,228.435273
8,peak_incremental_ram_mb,259.210938,1.236907,257.675115,260.746760
9,average_gpu_memory_mb,7300.917969,0.000000,7300.917969,7300.917969



Main values for the manuscript
------------------------------------------------------------------------------------------
latency_ms_per_image: 0.677 ± 0.010 (95% CI: 0.665–0.690)
peak_ram_mb: 6519.599 ± 1.589 (95% CI: 6517.626–6521.573)
peak_gpu_memory_mb: 7300.918 ± 0.000 (95% CI: 7300.918–7300.918)
average_gpu_utilization_percent: 61.473 ± 1.414 (95% CI: 59.718–63.228)
average_gpu_power_w: 31.798 ± 4.496 (95% CI: 26.215–37.381)


#genralization

In [12]:
#wild deepfake on dfc
test_results = model.evaluate(
    test_images,
    test_labels,
    batch_size=16,
    verbose=1,
    return_dict=True
)

print("\nTest results of DFC on wild deepfake dataset (EfficientNetV2_B0):")
for metric_name, metric_value in test_results.items():
    print(f"{metric_name}: {metric_value:.4f}")


1125/1125 [==============================] - 12s 10ms/step - loss: 2.8345 - accuracy: 0.4167 - precision: 0.7052 - recall: 0.3819 - roc_auc: 0.4345 - pr_auc: 0.7143

Test results of DFC on wild deepfake dataset (EfficientNetV2_B0):
loss: 2.8345
accuracy: 0.4167
precision: 0.7052
recall: 0.3819
roc_auc: 0.4345
pr_auc: 0.7143


In [17]:
#celeb on dfc
test_results = model.evaluate(
    test_celeb,
    test_labels,
    batch_size=16,
    verbose=1,
    return_dict=True
)

print("\nTest results of DFC on Celeb-DF(V2) dataset (EfficientNetV2_B0):")
for metric_name, metric_value in test_results.items():
    print(f"{metric_name}: {metric_value:.4f}")



373/373 [==============================] - 4s 11ms/step - loss: 2.9447 - accuracy: 0.3971 - precision: 0.8847 - recall: 0.3831 - roc_auc: 0.4525 - pr_auc: 0.8912

Test results of DFC on Celeb-DF(V2) dataset (EfficientNetV2_B0):
loss: 2.9447
accuracy: 0.3971
precision: 0.8847
recall: 0.3831
roc_auc: 0.4525
pr_auc: 0.8912


In [16]:
#FF++ on hog
test_results = model.evaluate(
    test_ff,
    test_ff_labels,
    batch_size=16,
    verbose=1,
    return_dict=True
)
print("\nTest results of dfc on FF++ dataset (EfficientNetV2_B0):")
for metric_name, metric_value in test_results.items():
    print(f"{metric_name}: {metric_value:.4f}")



155/155 [==============================] - 2s 11ms/step - loss: 2.6641 - accuracy: 0.5038 - precision: 0.3994 - recall: 0.3792 - roc_auc: 0.4777 - pr_auc: 0.4009

Test results of dfc on FF++ dataset (EfficientNetV2_B0):
loss: 2.6641
accuracy: 0.5038
precision: 0.3994
recall: 0.3792
roc_auc: 0.4777
pr_auc: 0.4009


# FF++

LOAD THE DATASET

In [4]:
import os, cv2, numpy as np

FINAL_ROOT = r'D:\thesis\ff_final'

def load_split(split, cls):
    base = os.path.join(FINAL_ROOT, split, cls)
    nested, ids = [], []
    for vid_id in sorted(os.listdir(base)):
        d = os.path.join(base, vid_id)
        frames = [cv2.imread(os.path.join(d, f)) for f in sorted(os.listdir(d))]
        if frames:
            nested.append(frames); ids.append(vid_id)
    return nested, ids

# Main splits
ff_real_train_f, ff_real_train_ids = load_split('train', 'real')
ff_fake_train_f, ff_fake_train_ids = load_split('train', 'fake')
ff_real_val_f,   ff_real_val_ids   = load_split('val',   'real')
ff_fake_val_f,   ff_fake_val_ids   = load_split('val',   'fake')
ff_real_test_f,  ff_real_test_ids  = load_split('test',  'real')
ff_fake_test_f,  ff_fake_test_ids  = load_split('test',  'fake')

print("Reloaded main splits. Example shape:", np.shape(ff_real_train_f[0][0]))  # (160,160,3)
print("Real train videos:", len(ff_real_train_f), "| Fake train videos:", len(ff_fake_train_f))
import numpy as np


def combine_split(real_videos, fake_videos):
    """
    Combine all frames from the real and fake video groups.

    Labels:
        0 = Real
        1 = Fake
    """

    real_frames = [
        frame
        for video in real_videos
        for frame in video
        if frame is not None
    ]

    fake_frames = [
        frame
        for video in fake_videos
        for frame in video
        if frame is not None
    ]

    if not real_frames:
        raise ValueError("No real frames found.")

    if not fake_frames:
        raise ValueError("No fake frames found.")

    real_frames = np.stack(real_frames).astype(np.uint8)
    fake_frames = np.stack(fake_frames).astype(np.uint8)

    images = np.concatenate(
        [real_frames, fake_frames],
        axis=0
    )

    real_labels = np.zeros(
        len(real_frames),
        dtype=np.uint8
    )

    fake_labels = np.ones(
        len(fake_frames),
        dtype=np.uint8
    )

    labels = np.concatenate(
        [real_labels, fake_labels],
        axis=0
    )

    return images, labels
# Training data
train_ff, train_ff_labels = combine_split(
    ff_real_train_f,
    ff_fake_train_f
)

# Validation data
val_ff, val_ff_labels = combine_split(
    ff_real_val_f,
    ff_fake_val_f
)

# Testing data
test_ff, test_ff_labels = combine_split(
    ff_real_test_f,
    ff_fake_test_f
)
print("\nTRAIN")
print("Images:", train_ff.shape)
print("Labels:", train_ff_labels.shape)
print("Real:", np.sum(train_ff_labels == 0))
print("Fake:", np.sum(train_ff_labels == 1))

print("\nVALIDATION")
print("Images:", val_ff.shape)
print("Labels:", val_ff_labels.shape)
print("Real:", np.sum(val_ff_labels == 0))
print("Fake:", np.sum(val_ff_labels == 1))

print("\nTEST")
print("Images:", test_ff.shape)
print("Labels:", test_ff_labels.shape)
print("Real:", np.sum(test_ff_labels == 0))
print("Fake:", np.sum(test_ff_labels == 1))

print("\nData types")
print("Train images:", train_ff.dtype)
print("Train labels:", train_ff_labels.dtype)

Reloaded main splits. Example shape: (160, 160, 3)
Real train videos: 517 | Fake train videos: 320

TRAIN
Images: (4595, 160, 160, 3)
Labels: (4595,)
Real: 2948
Fake: 1647

VALIDATION
Images: (948, 160, 160, 3)
Labels: (948,)
Real: 499
Fake: 449

TEST
Images: (2475, 160, 160, 3)
Labels: (2475,)
Real: 1444
Fake: 1031

Data types
Train images: uint8
Train labels: uint8


In [6]:
print("=== DATA LOADING ===")
start = monitor.get_stats()

=== DATA LOADING ===


In [4]:

# Train model
history = model.fit(
    train_ff, train_ff_labels,
    validation_data=(val_ff, val_ff_labels),
    epochs=10,
    batch_size=16,
    verbose=1
)

Epoch 1/10
288/288 [==============================] - 21s 47ms/step - loss: 0.3590 - accuracy: 0.8353 - precision: 0.8183 - recall: 0.6946 - roc_auc: 0.9061 - pr_auc: 0.8659 - val_loss: 0.2817 - val_accuracy: 0.8766 - val_precision: 0.8156 - val_recall: 0.9555 - val_roc_auc: 0.9693 - val_pr_auc: 0.9695
Epoch 2/10
288/288 [==============================] - 12s 43ms/step - loss: 0.1796 - accuracy: 0.9238 - precision: 0.9170 - recall: 0.8658 - roc_auc: 0.9775 - pr_auc: 0.9661 - val_loss: 0.2116 - val_accuracy: 0.9072 - val_precision: 0.8706 - val_recall: 0.9443 - val_roc_auc: 0.9756 - val_pr_auc: 0.9730
Epoch 3/10
288/288 [==============================] - 13s 43ms/step - loss: 0.1201 - accuracy: 0.9506 - precision: 0.9394 - recall: 0.9217 - roc_auc: 0.9893 - pr_auc: 0.9839 - val_loss: 0.2441 - val_accuracy: 0.9072 - val_precision: 0.8784 - val_recall: 0.9332 - val_roc_auc: 0.9752 - val_pr_auc: 0.9645
Epoch 4/10
288/288 [==============================] - 13s 44ms/step - loss: 0.0918 - acc

In [1]:
# ============================================================
# COMPLETE TEST EVALUATION
# Labels: 0 = Real, 1 = Fake
# ============================================================

import numpy as np

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    roc_curve,
    auc,
    confusion_matrix,
    classification_report,
    matthews_corrcoef
)

BATCH_SIZE = 16
DECISION_THRESHOLD = 0.50


# ------------------------------------------------------------
# 1. Test loss
# ------------------------------------------------------------

evaluation = model.evaluate(
    test_ff,
    test_ff_labels,
    batch_size=BATCH_SIZE,
    verbose=1,
    return_dict=True
)

test_loss = evaluation["loss"]


# ------------------------------------------------------------
# 2. Prediction probabilities and binary predictions
# ------------------------------------------------------------

test_probabilities = model.predict(
    test_ff,
    batch_size=BATCH_SIZE,
    verbose=1
).reshape(-1)

y_true = np.asarray(test_ff_labels).reshape(-1).astype(np.uint8)

y_pred = (
    test_probabilities >= DECISION_THRESHOLD
).astype(np.uint8)


# ------------------------------------------------------------
# 3. Confusion matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1]
)

tn, fp, fn, tp = cm.ravel()


# ------------------------------------------------------------
# 4. Threshold-dependent metrics
# ------------------------------------------------------------

accuracy = accuracy_score(y_true, y_pred)

balanced_accuracy = balanced_accuracy_score(
    y_true,
    y_pred
)

precision = precision_score(
    y_true,
    y_pred,
    pos_label=1,
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    pos_label=1,
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    pos_label=1,
    zero_division=0
)

specificity = (
    tn / (tn + fp)
    if (tn + fp) > 0
    else 0.0
)

false_positive_rate_at_05 = (
    fp / (fp + tn)
    if (fp + tn) > 0
    else 0.0
)

false_negative_rate_at_05 = (
    fn / (fn + tp)
    if (fn + tp) > 0
    else 0.0
)

mcc = matthews_corrcoef(
    y_true,
    y_pred
)


# ------------------------------------------------------------
# 5. ROC-AUC
# ------------------------------------------------------------

roc_auc = roc_auc_score(
    y_true,
    test_probabilities
)

fpr, tpr, roc_thresholds = roc_curve(
    y_true,
    test_probabilities,
    pos_label=1
)


# ------------------------------------------------------------
# 6. PR-AUC and Average Precision
# ------------------------------------------------------------

pr_precision, pr_recall, _ = precision_recall_curve(
    y_true,
    test_probabilities,
    pos_label=1
)

# Reverse because recall is normally returned in descending order
pr_auc = auc(
    pr_recall[::-1],
    pr_precision[::-1]
)

average_precision = average_precision_score(
    y_true,
    test_probabilities
)


# ------------------------------------------------------------
# 7. Equal Error Rate
# ------------------------------------------------------------

fnr_curve = 1.0 - tpr

# Remove non-finite thresholds such as infinity
valid_indices = np.where(
    np.isfinite(roc_thresholds)
)[0]

eer_index = valid_indices[
    np.argmin(
        np.abs(
            fpr[valid_indices]
            - fnr_curve[valid_indices]
        )
    )
]

eer = (
    fpr[eer_index]
    + fnr_curve[eer_index]
) / 2.0

eer_threshold = roc_thresholds[eer_index]


# ------------------------------------------------------------
# 8. Display all results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("EfficientNetB0 — COMPLETE TEST-SET EVALUATION")
print("=" * 70)

print(f"Number of test samples:     {len(y_true)}")
print(f"Decision threshold:         {DECISION_THRESHOLD:.4f}")
print(f"Test loss:                  {test_loss:.6f}")

print("\nMain evaluation metrics")
print("-" * 70)
print(f"Accuracy:                   {accuracy:.6f} ({accuracy*100:.2f}%)")
print(f"Precision:                  {precision:.6f} ({precision*100:.2f}%)")
print(f"Recall/Sensitivity:         {recall:.6f} ({recall*100:.2f}%)")
print(f"F1-score:                   {f1:.6f} ({f1*100:.2f}%)")
print(f"ROC-AUC:                    {roc_auc:.6f}")
print(f"PR-AUC:                     {pr_auc:.6f}")
print(f"Average Precision:          {average_precision:.6f}")
print(f"EER:                        {eer:.6f} ({eer*100:.2f}%)")
print(f"EER threshold:              {eer_threshold:.6f}")

print("\nAdditional evaluation metrics")
print("-" * 70)
print(f"Balanced accuracy:          {balanced_accuracy:.6f}")
print(f"Specificity:                {specificity:.6f}")
print(f"Matthews correlation:       {mcc:.6f}")
print(f"False-positive rate @ 0.5:  {false_positive_rate_at_05:.6f}")
print(f"False-negative rate @ 0.5:  {false_negative_rate_at_05:.6f}")

print("\nConfusion matrix")
print("-" * 70)
print("Rows = actual classes; columns = predicted classes")
print("Class order: [Real, Fake]")
print(cm)

print("\nConfusion-matrix values")
print("-" * 70)
print(f"True Negative  — Real predicted as Real: {tn}")
print(f"False Positive — Real predicted as Fake: {fp}")
print(f"False Negative — Fake predicted as Real: {fn}")
print(f"True Positive  — Fake predicted as Fake: {tp}")

print("\nClassification report")
print("-" * 70)

print(
    classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=["Real", "Fake"],
        digits=6,
        zero_division=0
    )
)


155/155 [==============================] - 3s 10ms/step

EfficientNetV2_B0 — COMPLETE TEST-SET EVALUATION
Number of test samples:     2475
Decision threshold:         0.5000
Test loss:                  0.554083

Main evaluation metrics
----------------------------------------------------------------------
Accuracy:                   0.842828 (84.28%)
Precision:                  0.737778 (73.78%)
Recall/Sensitivity:         0.966052 (96.61%)
F1-score:                   0.836623 (83.66%)
ROC-AUC:                    0.962078
PR-AUC:                     0.951561
Average Precision:          0.951585
EER:                        0.109025 (10.90%)
EER threshold:              0.971914

Additional evaluation metrics
----------------------------------------------------------------------
Balanced accuracy:          0.860450
Specificity:                0.754848
Matthews correlation:       0.713748
False-positive rate @ 0.5:  0.245152
False-negative rate @ 0.5:  0.033948

Confusion matrix
---------

In [2]:
test_results = model.evaluate(
    test_ff,
    test_ff_labels,
    batch_size=16,
    verbose=1,
    return_dict=True
)

print("\nTest results:")
for metric_name, metric_value in test_results.items():
    print(f"{metric_name}: {metric_value:.4f}")


155/155 [==============================] - 2s 12ms/step - loss: 0.5540 - accuracy: 0.8428 - precision: 0.7377 - recall: 0.9660 - roc_auc: 0.9620 - pr_auc: 0.9515

Test results:
loss: 0.5540
accuracy: 0.8428
precision: 0.7377
recall: 0.9660
roc_auc: 0.9620
pr_auc: 0.9515



In [8]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"RAM Used: {end['ram_mb'] - start['ram_mb']:.1f} MB")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 20.0%
RAM Used: -127.0 MB
Time Usage: 8.3 s
GPU Memory Used: 253.3 MB
Power Consumption: 93W


In [5]:
print("=== DATA LOADING ===")
start = monitor.get_stats()

=== DATA LOADING ===


In [7]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"RAM Used: {end['ram_mb'] - start['ram_mb']:.1f} MB")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 8.1%
RAM Used: 247.9 MB
Time Usage: 4.1 s
GPU Memory Used: 638.7 MB
Power Consumption: 93W


#save the model

In [ ]:
MODEL_PATH = (
    r"D:\thesis\results"
    r"\EfficientNetB0_ff_10epochs.h5"
)

model.save(
    MODEL_PATH,
    include_optimizer=True
)

print("Model saved successfully:")
print(MODEL_PATH)
# After training the model
model.save('EfficientNetB0_160_ff.h5')  # Saves the entire model to a file

Model saved successfully:
D:\thesis\results\Inception_V3_ff_10epochs.h5


#load the model

In [ ]:
import h5py
import tensorflow as tf

MODEL_PATH = ( r"D:\thesis\results" r"\EfficientNetB0_ff_10epochs.h5")


model = tf.keras.models.load_model(MODEL_PATH,compile=False)

In [8]:
# ============================================================
# REPEATED INFERENCE RESOURCE PROFILING
# ============================================================

import os
import gc
import time
import threading
import numpy as np
import pandas as pd
import psutil

from scipy.stats import t

from pynvml import (
    nvmlInit,
    nvmlShutdown,
    nvmlDeviceGetHandleByIndex,
    nvmlDeviceGetMemoryInfo,
    nvmlDeviceGetUtilizationRates,
    nvmlDeviceGetPowerUsage,
    NVMLError
)


BATCH_SIZE = 16
NUMBER_OF_RUNS = 5
SAMPLING_INTERVAL = 0.1     # Sample every 100 milliseconds
COOLDOWN_SECONDS = 5
GPU_INDEX = 0


class ResourceMonitor:
    """
    Continuously measures resource consumption during one inference run.

    RAM:
        Current Python process resident memory (RSS).

    GPU memory:
        Total device memory currently used, with the pre-inference
        baseline subtracted for incremental measurements.

    CPU:
        Current Python-process CPU utilization normalized by the
        number of logical CPU cores.

    Power:
        NVIDIA GPU board power reported by NVML.
    """

    def __init__(
        self,
        interval=0.1,
        gpu_index=0
    ):
        self.interval = interval
        self.process = psutil.Process(os.getpid())

        self.logical_cpu_count = (
            psutil.cpu_count(logical=True) or 1
        )

        self.stop_event = threading.Event()
        self.samples = []
        self.thread = None

        nvmlInit()
        self.gpu_handle = nvmlDeviceGetHandleByIndex(
            gpu_index
        )

    def _read_gpu_power(self):
        try:
            return (
                nvmlDeviceGetPowerUsage(
                    self.gpu_handle
                ) / 1000.0
            )
        except NVMLError:
            return np.nan

    def _collect_sample(self):
        timestamp = time.perf_counter()

        # Process RAM: resident set size
        ram_mb = (
            self.process.memory_info().rss
            / (1024 ** 2)
        )

        # Process CPU percentage can exceed 100% on multicore systems.
        # Normalize it to an approximate 0–100% scale.
        process_cpu_raw = self.process.cpu_percent(
            interval=None
        )

        process_cpu_normalized = (
            process_cpu_raw
            / self.logical_cpu_count
        )

        gpu_memory = nvmlDeviceGetMemoryInfo(
            self.gpu_handle
        )

        gpu_memory_used_mb = (
            gpu_memory.used
            / (1024 ** 2)
        )

        gpu_utilization = nvmlDeviceGetUtilizationRates(
            self.gpu_handle
        ).gpu

        gpu_power_w = self._read_gpu_power()

        self.samples.append({
            "timestamp": timestamp,
            "ram_mb": ram_mb,
            "cpu_percent": process_cpu_normalized,
            "gpu_memory_mb": gpu_memory_used_mb,
            "gpu_utilization_percent": gpu_utilization,
            "gpu_power_w": gpu_power_w
        })

    def _sampling_loop(self):
        while not self.stop_event.is_set():
            try:
                self._collect_sample()
            except Exception as error:
                print("Monitoring warning:", error)

            self.stop_event.wait(self.interval)

    def start(self):
        # Initialize CPU counters
        self.process.cpu_percent(interval=None)

        # First sample is the pre-inference baseline
        self._collect_sample()

        self.thread = threading.Thread(
            target=self._sampling_loop,
            daemon=True
        )

        self.thread.start()

    def stop(
        self,
        elapsed_seconds,
        number_of_images
    ):
        self.stop_event.set()

        if self.thread is not None:
            self.thread.join()

        # Capture one final sample
        try:
            self._collect_sample()
        except Exception:
            pass

        nvmlShutdown()

        data = pd.DataFrame(self.samples)

        baseline_ram = data["ram_mb"].iloc[0]
        baseline_gpu_memory = data[
            "gpu_memory_mb"
        ].iloc[0]

        peak_ram = data["ram_mb"].max()
        average_ram = data["ram_mb"].mean()

        peak_gpu_memory = data[
            "gpu_memory_mb"
        ].max()

        average_gpu_memory = data[
            "gpu_memory_mb"
        ].mean()

        peak_incremental_ram = max(
            0.0,
            peak_ram - baseline_ram
        )

        average_incremental_ram = max(
            0.0,
            average_ram - baseline_ram
        )

        peak_incremental_gpu_memory = max(
            0.0,
            peak_gpu_memory - baseline_gpu_memory
        )

        average_incremental_gpu_memory = max(
            0.0,
            average_gpu_memory - baseline_gpu_memory
        )

        # Estimate GPU energy using power integration
        valid_power = data.dropna(
            subset=["gpu_power_w"]
        )

        if len(valid_power) >= 2:
            relative_times = (
                valid_power["timestamp"].to_numpy()
                - valid_power["timestamp"].iloc[0]
            )

            energy_joules = np.trapz(
                valid_power["gpu_power_w"].to_numpy(),
                relative_times
            )

            energy_wh = energy_joules / 3600.0
        else:
            energy_wh = np.nan

        return {
            "elapsed_time_s": elapsed_seconds,

            "latency_ms_per_image": (
                elapsed_seconds
                / number_of_images
                * 1000.0
            ),

            "throughput_images_per_s": (
                number_of_images
                / elapsed_seconds
            ),

            "average_cpu_percent": (
                data["cpu_percent"].mean()
            ),

            "peak_cpu_percent": (
                data["cpu_percent"].max()
            ),

            "baseline_ram_mb": baseline_ram,
            "average_ram_mb": average_ram,
            "peak_ram_mb": peak_ram,

            "average_incremental_ram_mb": (
                average_incremental_ram
            ),

            "peak_incremental_ram_mb": (
                peak_incremental_ram
            ),

            "baseline_gpu_memory_mb": (
                baseline_gpu_memory
            ),

            "average_gpu_memory_mb": (
                average_gpu_memory
            ),

            "peak_gpu_memory_mb": (
                peak_gpu_memory
            ),

            "average_incremental_gpu_memory_mb": (
                average_incremental_gpu_memory
            ),

            "peak_incremental_gpu_memory_mb": (
                peak_incremental_gpu_memory
            ),

            "average_gpu_utilization_percent": (
                data["gpu_utilization_percent"].mean()
            ),

            "peak_gpu_utilization_percent": (
                data["gpu_utilization_percent"].max()
            ),

            "average_gpu_power_w": (
                data["gpu_power_w"].mean()
            ),

            "peak_gpu_power_w": (
                data["gpu_power_w"].max()
            ),

            "gpu_energy_wh": energy_wh
        }


# ------------------------------------------------------------
# GPU/model warm-up
# ------------------------------------------------------------

warmup_count = min(
    len(test_ff),
    BATCH_SIZE * 3
)

print("Performing warm-up inference...")

_ = model.predict(
    test_ff[:warmup_count],
    batch_size=BATCH_SIZE,
    verbose=0
)

print("Warm-up completed.")


# ------------------------------------------------------------
# Five repeated inference runs
# ------------------------------------------------------------

run_results = []

for run_number in range(
    1,
    NUMBER_OF_RUNS + 1
):
    print(
        f"\nStarting resource run "
        f"{run_number}/{NUMBER_OF_RUNS}"
    )

    gc.collect()
    time.sleep(COOLDOWN_SECONDS)

    monitor = ResourceMonitor(
        interval=SAMPLING_INTERVAL,
        gpu_index=GPU_INDEX
    )

    monitor.start()

    start_time = time.perf_counter()

    predictions = model.predict(
        test_ff,
        batch_size=BATCH_SIZE,
        verbose=0
    )

    elapsed_time = (
        time.perf_counter()
        - start_time
    )

    # Access the result to ensure it is materialized
    _ = float(predictions[-1].reshape(-1)[0])

    run_summary = monitor.stop(
        elapsed_seconds=elapsed_time,
        number_of_images=len(test_ff)
    )

    run_summary["run"] = run_number
    run_results.append(run_summary)

    print(
        f"Run {run_number}: "
        f"{elapsed_time:.2f} seconds, "
        f"{run_summary['latency_ms_per_image']:.4f} ms/image"
    )

    del predictions


results_df = pd.DataFrame(run_results)

print("\nIndividual runs:")
display(results_df)

C:\Users\aneek\AppData\Local\Temp\ipykernel_38032\115312294.py:15: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  from pynvml import (


Performing warm-up inference...
Warm-up completed.

Starting resource run 1/5
Run 1: 2.83 seconds, 1.1435 ms/image

Starting resource run 2/5
Run 2: 1.69 seconds, 0.6834 ms/image

Starting resource run 3/5
Run 3: 1.71 seconds, 0.6894 ms/image

Starting resource run 4/5
Run 4: 1.68 seconds, 0.6772 ms/image

Starting resource run 5/5
Run 5: 1.64 seconds, 0.6643 ms/image

Individual runs:


,elapsed_time_s,latency_ms_per_image,throughput_images_per_s,average_cpu_percent,peak_cpu_percent,baseline_ram_mb,average_ram_mb,peak_ram_mb,average_incremental_ram_mb,peak_incremental_ram_mb,...,average_gpu_memory_mb,peak_gpu_memory_mb,average_incremental_gpu_memory_mb,peak_incremental_gpu_memory_mb,average_gpu_utilization_percent,peak_gpu_utilization_percent,average_gpu_power_w,peak_gpu_power_w,gpu_energy_wh,run
0,2.830098,1.143474,874.528051,3.031250,4.031250,5212.125000,5421.156250,5461.601562,209.031250,249.476562,...,6983.027344,6983.027344,0.0,0.0,55.851852,97,23.926593,63.803,0.018149,1
1,1.691361,0.683378,1463.318507,2.740993,3.584375,5243.343750,5431.325827,5463.144531,187.982077,219.800781,...,6983.027344,6983.027344,0.0,0.0,68.235294,89,29.346118,68.032,0.013809,2
2,1.706232,0.689387,1450.564583,2.908681,4.031250,5243.476562,5432.675347,5463.203125,189.198785,219.726562,...,6983.027344,6983.027344,0.0,0.0,65.777778,98,30.228722,63.578,0.014301,3
3,1.676130,0.677224,1476.615599,2.679228,3.584375,5243.523438,5431.391544,5464.179688,187.868107,220.656250,...,6983.027344,6983.027344,0.0,0.0,67.470588,83,29.083294,63.329,0.013502,4
4,1.644177,0.664314,1505.312659,3.073713,5.512500,5243.558594,5432.046186,5463.257812,188.487592,219.699219,...,6983.027344,6983.027344,0.0,0.0,64.823529,82,30.639235,62.062,0.013995,5


In [9]:
# ============================================================RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# ============================================================

metrics_to_report = [
    "elapsed_time_s",
    "latency_ms_per_image",
    "throughput_images_per_s",

    "average_cpu_percent",
    "peak_cpu_percent",

    "average_ram_mb",
    "peak_ram_mb",
    "average_incremental_ram_mb",
    "peak_incremental_ram_mb",

    "average_gpu_memory_mb",
    "peak_gpu_memory_mb",
    "average_incremental_gpu_memory_mb",
    "peak_incremental_gpu_memory_mb",

    "average_gpu_utilization_percent",
    "peak_gpu_utilization_percent",

    "average_gpu_power_w",
    "peak_gpu_power_w",
    "gpu_energy_wh"
]


summary_rows = []

number_of_runs = len(results_df)

for metric in metrics_to_report:
    values = results_df[metric].dropna()

    mean_value = values.mean()
    standard_deviation = values.std(ddof=1)

    if len(values) > 1:
        critical_t = t.ppf(
            0.975,
            df=len(values) - 1
        )

        confidence_half_width = (
            critical_t
            * standard_deviation
            / np.sqrt(len(values))
        )
    else:
        confidence_half_width = np.nan

    summary_rows.append({
        "Metric": metric,
        "Mean": mean_value,
        "Standard Deviation": standard_deviation,
        "95% CI Lower": (
            mean_value - confidence_half_width
        ),
        "95% CI Upper": (
            mean_value + confidence_half_width
        )
    })


resource_summary = pd.DataFrame(summary_rows)

print("\n" + "=" * 90)
print("EfficientNetB0 RESOURCE CONSUMPTION — FIVE INFERENCE RUNS")
print("=" * 90)

display(resource_summary)


print("\nMain values for the manuscript")
print("-" * 90)

for metric in [
    "latency_ms_per_image",
    "peak_ram_mb",
    "peak_gpu_memory_mb",
    "average_gpu_utilization_percent",
    "average_gpu_power_w"
]:
    row = resource_summary[
        resource_summary["Metric"] == metric
    ].iloc[0]

    print(
        f"{metric}: "
        f"{row['Mean']:.3f} ± "
        f"{row['Standard Deviation']:.3f} "
        f"(95% CI: "
        f"{row['95% CI Lower']:.3f}–"
        f"{row['95% CI Upper']:.3f})"
    )


EfficientNetB0 RESOURCE CONSUMPTION — FIVE INFERENCE RUNS


,Metric,Mean,Standard Deviation,95% CI Lower,95% CI Upper
0,elapsed_time_s,1.909600,0.515088,1.270034,2.549165
1,latency_ms_per_image,0.771555,0.208116,0.513145,1.029966
2,throughput_images_per_s,1354.067880,268.839491,1020.259563,1687.876197
3,average_cpu_percent,2.886773,0.173657,2.671149,3.102397
4,peak_cpu_percent,4.148750,0.794428,3.162337,5.135163
5,average_ram_mb,5429.719031,4.818085,5423.736588,5435.701473
6,peak_ram_mb,5463.077344,0.928170,5461.924869,5464.229819
7,average_incremental_ram_mb,192.513562,9.248578,181.029935,203.997189
8,peak_incremental_ram_mb,225.871875,13.201409,209.480161,242.263589
9,average_gpu_memory_mb,6983.027344,0.000000,6983.027344,6983.027344



Main values for the manuscript
------------------------------------------------------------------------------------------
latency_ms_per_image: 0.772 ± 0.208 (95% CI: 0.513–1.030)
peak_ram_mb: 5463.077 ± 0.928 (95% CI: 5461.925–5464.230)
peak_gpu_memory_mb: 6983.027 ± 0.000 (95% CI: 6983.027–6983.027)
average_gpu_utilization_percent: 64.432 ± 4.982 (95% CI: 58.246–70.618)
average_gpu_power_w: 28.645 ± 2.713 (95% CI: 25.277–32.013)


In [9]:
# ============================================================
# REPEATED INFERENCE RESOURCE PROFILING
# ============================================================

import os
import gc
import time
import threading
import numpy as np
import pandas as pd
import psutil

from scipy.stats import t

from pynvml import (
    nvmlInit,
    nvmlShutdown,
    nvmlDeviceGetHandleByIndex,
    nvmlDeviceGetMemoryInfo,
    nvmlDeviceGetUtilizationRates,
    nvmlDeviceGetPowerUsage,
    NVMLError
)


BATCH_SIZE = 16
NUMBER_OF_RUNS = 5
SAMPLING_INTERVAL = 0.1     # Sample every 100 milliseconds
COOLDOWN_SECONDS = 5
GPU_INDEX = 0


class ResourceMonitor:
    """
    Continuously measures resource consumption during one inference run.

    RAM:
        Current Python process resident memory (RSS).

    GPU memory:
        Total device memory currently used, with the pre-inference
        baseline subtracted for incremental measurements.

    CPU:
        Current Python-process CPU utilization normalized by the
        number of logical CPU cores.

    Power:
        NVIDIA GPU board power reported by NVML.
    """

    def __init__(
        self,
        interval=0.1,
        gpu_index=0
    ):
        self.interval = interval
        self.process = psutil.Process(os.getpid())

        self.logical_cpu_count = (
            psutil.cpu_count(logical=True) or 1
        )

        self.stop_event = threading.Event()
        self.samples = []
        self.thread = None

        nvmlInit()
        self.gpu_handle = nvmlDeviceGetHandleByIndex(
            gpu_index
        )

    def _read_gpu_power(self):
        try:
            return (
                nvmlDeviceGetPowerUsage(
                    self.gpu_handle
                ) / 1000.0
            )
        except NVMLError:
            return np.nan

    def _collect_sample(self):
        timestamp = time.perf_counter()

        # Process RAM: resident set size
        ram_mb = (
            self.process.memory_info().rss
            / (1024 ** 2)
        )

        # Process CPU percentage can exceed 100% on multicore systems.
        # Normalize it to an approximate 0–100% scale.
        process_cpu_raw = self.process.cpu_percent(
            interval=None
        )

        process_cpu_normalized = (
            process_cpu_raw
            / self.logical_cpu_count
        )

        gpu_memory = nvmlDeviceGetMemoryInfo(
            self.gpu_handle
        )

        gpu_memory_used_mb = (
            gpu_memory.used
            / (1024 ** 2)
        )

        gpu_utilization = nvmlDeviceGetUtilizationRates(
            self.gpu_handle
        ).gpu

        gpu_power_w = self._read_gpu_power()

        self.samples.append({
            "timestamp": timestamp,
            "ram_mb": ram_mb,
            "cpu_percent": process_cpu_normalized,
            "gpu_memory_mb": gpu_memory_used_mb,
            "gpu_utilization_percent": gpu_utilization,
            "gpu_power_w": gpu_power_w
        })

    def _sampling_loop(self):
        while not self.stop_event.is_set():
            try:
                self._collect_sample()
            except Exception as error:
                print("Monitoring warning:", error)

            self.stop_event.wait(self.interval)

    def start(self):
        # Initialize CPU counters
        self.process.cpu_percent(interval=None)

        # First sample is the pre-inference baseline
        self._collect_sample()

        self.thread = threading.Thread(
            target=self._sampling_loop,
            daemon=True
        )

        self.thread.start()

    def stop(
        self,
        elapsed_seconds,
        number_of_images
    ):
        self.stop_event.set()

        if self.thread is not None:
            self.thread.join()

        # Capture one final sample
        try:
            self._collect_sample()
        except Exception:
            pass

        nvmlShutdown()

        data = pd.DataFrame(self.samples)

        baseline_ram = data["ram_mb"].iloc[0]
        baseline_gpu_memory = data[
            "gpu_memory_mb"
        ].iloc[0]

        peak_ram = data["ram_mb"].max()
        average_ram = data["ram_mb"].mean()

        peak_gpu_memory = data[
            "gpu_memory_mb"
        ].max()

        average_gpu_memory = data[
            "gpu_memory_mb"
        ].mean()

        peak_incremental_ram = max(
            0.0,
            peak_ram - baseline_ram
        )

        average_incremental_ram = max(
            0.0,
            average_ram - baseline_ram
        )

        peak_incremental_gpu_memory = max(
            0.0,
            peak_gpu_memory - baseline_gpu_memory
        )

        average_incremental_gpu_memory = max(
            0.0,
            average_gpu_memory - baseline_gpu_memory
        )

        # Estimate GPU energy using power integration
        valid_power = data.dropna(
            subset=["gpu_power_w"]
        )

        if len(valid_power) >= 2:
            relative_times = (
                valid_power["timestamp"].to_numpy()
                - valid_power["timestamp"].iloc[0]
            )

            energy_joules = np.trapz(
                valid_power["gpu_power_w"].to_numpy(),
                relative_times
            )

            energy_wh = energy_joules / 3600.0
        else:
            energy_wh = np.nan

        return {
            "elapsed_time_s": elapsed_seconds,

            "latency_ms_per_image": (
                elapsed_seconds
                / number_of_images
                * 1000.0
            ),

            "throughput_images_per_s": (
                number_of_images
                / elapsed_seconds
            ),

            "average_cpu_percent": (
                data["cpu_percent"].mean()
            ),

            "peak_cpu_percent": (
                data["cpu_percent"].max()
            ),

            "baseline_ram_mb": baseline_ram,
            "average_ram_mb": average_ram,
            "peak_ram_mb": peak_ram,

            "average_incremental_ram_mb": (
                average_incremental_ram
            ),

            "peak_incremental_ram_mb": (
                peak_incremental_ram
            ),

            "baseline_gpu_memory_mb": (
                baseline_gpu_memory
            ),

            "average_gpu_memory_mb": (
                average_gpu_memory
            ),

            "peak_gpu_memory_mb": (
                peak_gpu_memory
            ),

            "average_incremental_gpu_memory_mb": (
                average_incremental_gpu_memory
            ),

            "peak_incremental_gpu_memory_mb": (
                peak_incremental_gpu_memory
            ),

            "average_gpu_utilization_percent": (
                data["gpu_utilization_percent"].mean()
            ),

            "peak_gpu_utilization_percent": (
                data["gpu_utilization_percent"].max()
            ),

            "average_gpu_power_w": (
                data["gpu_power_w"].mean()
            ),

            "peak_gpu_power_w": (
                data["gpu_power_w"].max()
            ),

            "gpu_energy_wh": energy_wh
        }


# ------------------------------------------------------------
# GPU/model warm-up
# ------------------------------------------------------------

warmup_count = min(
    len(test_ff),
    BATCH_SIZE * 3
)

print("Performing warm-up inference...")

_ = model.predict(
    test_ff[:warmup_count],
    batch_size=BATCH_SIZE,
    verbose=0
)

print("Warm-up completed.")


# ------------------------------------------------------------
# Five repeated inference runs
# ------------------------------------------------------------

run_results = []

for run_number in range(
    1,
    NUMBER_OF_RUNS + 1
):
    print(
        f"\nStarting resource run "
        f"{run_number}/{NUMBER_OF_RUNS}"
    )

    gc.collect()
    time.sleep(COOLDOWN_SECONDS)

    monitor = ResourceMonitor(
        interval=SAMPLING_INTERVAL,
        gpu_index=GPU_INDEX
    )

    monitor.start()

    start_time = time.perf_counter()

    predictions = model.predict(
        test_ff,
        batch_size=BATCH_SIZE,
        verbose=0
    )

    elapsed_time = (
        time.perf_counter()
        - start_time
    )

    # Access the result to ensure it is materialized
    _ = float(predictions[-1].reshape(-1)[0])

    run_summary = monitor.stop(
        elapsed_seconds=elapsed_time,
        number_of_images=len(test_ff)
    )

    run_summary["run"] = run_number
    run_results.append(run_summary)

    print(
        f"Run {run_number}: "
        f"{elapsed_time:.2f} seconds, "
        f"{run_summary['latency_ms_per_image']:.4f} ms/image"
    )

    del predictions


results_df = pd.DataFrame(run_results)

print("\nIndividual runs:")
display(results_df)

Performing warm-up inference...


C:\Users\aneek\AppData\Local\Temp\ipykernel_11320\115312294.py:15: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  from pynvml import (


Warm-up completed.

Starting resource run 1/5
Run 1: 1.71 seconds, 0.6898 ms/image

Starting resource run 2/5
Run 2: 1.58 seconds, 0.6401 ms/image

Starting resource run 3/5
Run 3: 1.64 seconds, 0.6625 ms/image

Starting resource run 4/5
Run 4: 1.63 seconds, 0.6580 ms/image

Starting resource run 5/5
Run 5: 1.67 seconds, 0.6729 ms/image

Individual runs:


,elapsed_time_s,latency_ms_per_image,throughput_images_per_s,average_cpu_percent,peak_cpu_percent,baseline_ram_mb,average_ram_mb,peak_ram_mb,average_incremental_ram_mb,peak_incremental_ram_mb,...,average_gpu_memory_mb,peak_gpu_memory_mb,average_incremental_gpu_memory_mb,peak_incremental_gpu_memory_mb,average_gpu_utilization_percent,peak_gpu_utilization_percent,average_gpu_power_w,peak_gpu_power_w,gpu_energy_wh,run
0,1.707371,0.689847,1449.597410,2.843934,4.478125,5236.847656,5420.678998,5456.613281,183.831342,219.765625,...,6987.777344,6987.777344,0.0,0.0,61.352941,94,31.859588,40.514,0.015228,1
1,1.584268,0.640108,1562.235387,3.140039,8.512500,5238.433594,5421.954102,5458.156250,183.520508,219.722656,...,6987.777344,6987.777344,0.0,0.0,64.437500,78,34.006812,44.685,0.015386,2
2,1.639640,0.662481,1509.477966,2.744922,3.993750,5238.593750,5421.753662,5458.382812,183.159912,219.789062,...,6987.777344,6987.777344,0.0,0.0,61.250000,95,32.214875,41.856,0.014963,3
3,1.628583,0.658013,1519.726324,2.722852,4.296875,5238.828125,5421.532227,5458.742188,182.704102,219.914062,...,6987.777344,6987.777344,0.0,0.0,62.500000,94,33.074687,42.392,0.015244,4
4,1.665491,0.672926,1486.048437,2.700368,4.156250,5239.773438,5423.300092,5459.792969,183.526654,220.019531,...,6987.777344,6987.777344,0.0,0.0,67.294118,84,32.384588,44.944,0.015099,5


In [10]:
# ============================================================RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# ============================================================

metrics_to_report = [
    "elapsed_time_s",
    "latency_ms_per_image",
    "throughput_images_per_s",

    "average_cpu_percent",
    "peak_cpu_percent",

    "average_ram_mb",
    "peak_ram_mb",
    "average_incremental_ram_mb",
    "peak_incremental_ram_mb",

    "average_gpu_memory_mb",
    "peak_gpu_memory_mb",
    "average_incremental_gpu_memory_mb",
    "peak_incremental_gpu_memory_mb",

    "average_gpu_utilization_percent",
    "peak_gpu_utilization_percent",

    "average_gpu_power_w",
    "peak_gpu_power_w",
    "gpu_energy_wh"
]


summary_rows = []

number_of_runs = len(results_df)

for metric in metrics_to_report:
    values = results_df[metric].dropna()

    mean_value = values.mean()
    standard_deviation = values.std(ddof=1)

    if len(values) > 1:
        critical_t = t.ppf(
            0.975,
            df=len(values) - 1
        )

        confidence_half_width = (
            critical_t
            * standard_deviation
            / np.sqrt(len(values))
        )
    else:
        confidence_half_width = np.nan

    summary_rows.append({
        "Metric": metric,
        "Mean": mean_value,
        "Standard Deviation": standard_deviation,
        "95% CI Lower": (
            mean_value - confidence_half_width
        ),
        "95% CI Upper": (
            mean_value + confidence_half_width
        )
    })


resource_summary = pd.DataFrame(summary_rows)

print("\n" + "=" * 90)
print("EfficientNetV2_B0 RESOURCE CONSUMPTION — FIVE INFERENCE RUNS")
print("=" * 90)

display(resource_summary)


print("\nMain values for the manuscript")
print("-" * 90)

for metric in [
    "latency_ms_per_image",
    "peak_ram_mb",
    "peak_gpu_memory_mb",
    "average_gpu_utilization_percent",
    "average_gpu_power_w"
]:
    row = resource_summary[
        resource_summary["Metric"] == metric
    ].iloc[0]

    print(
        f"{metric}: "
        f"{row['Mean']:.3f} ± "
        f"{row['Standard Deviation']:.3f} "
        f"(95% CI: "
        f"{row['95% CI Lower']:.3f}–"
        f"{row['95% CI Upper']:.3f})"
    )


EfficientNetV2_B0 RESOURCE CONSUMPTION — FIVE INFERENCE RUNS


,Metric,Mean,Standard Deviation,95% CI Lower,95% CI Upper
0,elapsed_time_s,1.645070,0.045543,1.588522,1.701619
1,latency_ms_per_image,0.664675,0.018401,0.641827,0.687523
2,throughput_images_per_s,1505.417105,41.654967,1453.695632,1557.138578
3,average_cpu_percent,2.830423,0.181550,2.604999,3.055846
4,peak_cpu_percent,5.087500,1.922925,2.699873,7.475127
5,average_ram_mb,5421.843816,0.948014,5420.666702,5423.020930
6,peak_ram_mb,5458.337500,1.150000,5456.909587,5459.765413
7,average_incremental_ram_mb,183.348504,0.431647,182.812543,183.884464
8,peak_incremental_ram_mb,219.842187,0.122054,219.690638,219.993737
9,average_gpu_memory_mb,6987.777344,0.000000,6987.777344,6987.777344



Main values for the manuscript
------------------------------------------------------------------------------------------
latency_ms_per_image: 0.665 ± 0.018 (95% CI: 0.642–0.688)
peak_ram_mb: 5458.337 ± 1.150 (95% CI: 5456.910–5459.765)
peak_gpu_memory_mb: 6987.777 ± 0.000 (95% CI: 6987.777–6987.777)
average_gpu_utilization_percent: 63.367 ± 2.543 (95% CI: 60.210–66.524)
average_gpu_power_w: 32.708 ± 0.850 (95% CI: 31.653–33.763)


#gernalization

In [12]:
#wild deepfake on ff
test_results = model.evaluate(
    test_images,
    test_labels,
    batch_size=16,
    verbose=1,
    return_dict=True
)

print("\nTest results of FF++ on wild deepfake dataset (EfficientNetV2_B0):")
for metric_name, metric_value in test_results.items():
    print(f"{metric_name}: {metric_value:.4f}")


1125/1125 [==============================] - 12s 11ms/step - loss: 0.9100 - accuracy: 0.3033 - precision: 0.7043 - recall: 0.1226 - roc_auc: 0.4999 - pr_auc: 0.7422

Test results of FF++ on wild deepfake dataset (EfficientNetV2_B0):
loss: 0.9100
accuracy: 0.3033
precision: 0.7043
recall: 0.1226
roc_auc: 0.4999
pr_auc: 0.7422


In [14]:
#celeb on ff
test_results = model.evaluate(
    test_celeb,
    test_labels,
    batch_size=16,
    verbose=1,
    return_dict=True
)

print("\nTest results of FF++ on Celeb-df(v2) dataset (EfficientNetV2_B0):")
for metric_name, metric_value in test_results.items():
    print(f"{metric_name}: {metric_value:.4f}")

373/373 [==============================] - 4s 12ms/step - loss: 0.9016 - accuracy: 0.1902 - precision: 0.8779 - recall: 0.1213 - roc_auc: 0.4921 - pr_auc: 0.8971

Test results of FF++ on Celeb-df(v2) dataset (EfficientNetV2_B0):
loss: 0.9016
accuracy: 0.1902
precision: 0.8779
recall: 0.1213
roc_auc: 0.4921
pr_auc: 0.8971


In [16]:
#DFC on ff
test_results = model.evaluate(
    test_hog,
    test_labels,
    batch_size=16,
    verbose=1,
    return_dict=True
)

print("\nTest results of FF++ on DFC dataset (EfficientNetV2_B0):")
for metric_name, metric_value in test_results.items():
    print(f"{metric_name}: {metric_value:.4f}")


188/188 [==============================] - 4s 13ms/step - loss: 1.0923 - accuracy: 0.5037 - precision: 0.5140 - recall: 0.1347 - roc_auc: 0.5145 - pr_auc: 0.5084

Test results of FF++ on DFC dataset (EfficientNetV2_B0):
loss: 1.0923
accuracy: 0.5037
precision: 0.5140
recall: 0.1347
roc_auc: 0.5145
pr_auc: 0.5084
